In [ ]:
import numpy as np
import pandas as pd
import scipy
import scipy.stats
import scipy.optimize
import matplotlib.pyplot as plt
from IPython.display import display

# ---------- global settings ----------

seed = 42
rng = np.random.default_rng(seed)

Rmax = 1.0
response_sd_default = 0.18

num_x_default = 301
num_r_default = 251

n_starts_default = 12
maxiter_default = 150

n_steps_main = 5000
burn_in_main = 500

# gamma now means bias toward the shared target:
# gamma = 0 -> naive local adaptive
# gamma = 1 -> full target
gamma_grid = np.linspace(0.0, 1.0, 9)

p_switch_main = 0.01
seed_list = [0, 1, 2, 3]

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
# ---------- contexts ----------

contexts_mean = {
    0: {"name": "low-mean",  "mu": -2.0, "sigma": 1.0},
    1: {"name": "high-mean", "mu":  2.0, "sigma": 1.0},
}

contexts_var = {
    0: {"name": "low-var",  "mu": 0.0, "sigma": 0.7},
    1: {"name": "high-var", "mu": 0.0, "sigma": 1.8},
}
# ---------- core one-neuron helpers ----------

def logistic(x, a, b, Rmax=Rmax):
    return Rmax / (1 + np.exp(-a * (x - b)))


def logistic_derivative(x, a, b, Rmax=Rmax):
    y = logistic(x, a, b, Rmax=Rmax)
    return a * y * (1 - y / Rmax)


def make_x_grid(contexts, n_std=5, num_x=num_x_default):
    mus = [c["mu"] for c in contexts.values()]
    sigmas = [c["sigma"] for c in contexts.values()]

    x_min = min(mu - n_std * sigma for mu, sigma in zip(mus, sigmas))
    x_max = max(mu + n_std * sigma for mu, sigma in zip(mus, sigmas))

    pad = 0.5
    return np.linspace(x_min - pad, x_max + pad, num_x)


def normal_pmf_on_grid(x_grid, mu, sigma):
    px = scipy.stats.norm.pdf(x_grid, loc=mu, scale=sigma)
    px = np.maximum(px, 1e-300)
    px /= px.sum()
    return px


def mixture_prior_pmf(contexts, x_grid, weights=None):
    keys = list(contexts.keys())

    if weights is None:
        weights = {k: 1 / len(keys) for k in keys}

    px = np.zeros_like(x_grid, dtype=float)

    for k in keys:
        px += weights[k] * normal_pmf_on_grid(
            x_grid,
            contexts[k]["mu"],
            contexts[k]["sigma"]
        )

    px = np.maximum(px, 1e-300)
    px /= px.sum()
    return px


def posterior_stats_from_prior_one_neuron(
    x_grid,
    px,
    a,
    b,
    response_sd=response_sd_default,
    Rmax=Rmax,
    num_r=num_r_default,
):
    mean_responses = logistic(x_grid, a, b, Rmax=Rmax)

    r_min = mean_responses.min() - 4 * response_sd
    r_max = mean_responses.max() + 4 * response_sd
    r_grid = np.linspace(r_min, r_max, num_r)

    # continuous response density on a common grid
    pr_given_x_density = scipy.stats.norm.pdf(
        r_grid[None, :],
        loc=mean_responses[:, None],
        scale=response_sd
    )
    pr_given_x_density = np.maximum(pr_given_x_density, 1e-300)

    # discrete mass approximation on the chosen r-grid
    pr_given_x = pr_given_x_density / pr_given_x_density.sum(axis=1, keepdims=True)

    p_r = np.sum(px[:, None] * pr_given_x, axis=0)
    p_r = np.maximum(p_r, 1e-300)
    p_r /= p_r.sum()

    posterior = (pr_given_x * px[:, None]) / p_r[None, :]
    posterior = np.maximum(posterior, 1e-300)
    posterior /= posterior.sum(axis=0, keepdims=True)

    posterior_mean = np.sum(posterior * x_grid[:, None], axis=0)

    bayes_mse = np.sum(
        px[:, None] * pr_given_x * (posterior_mean[None, :] - x_grid[:, None]) ** 2
    )

    return {
        "x_grid": x_grid,
        "px": px,
        "a": a,
        "b": b,
        "mean_responses": mean_responses,
        "r_grid": r_grid,
        "pr_given_x_density": pr_given_x_density,
        "pr_given_x": pr_given_x,
        "p_r": p_r,
        "posterior": posterior,
        "posterior_mean": posterior_mean,
        "bayes_mse": bayes_mse,
    }


def posterior_stats_one_neuron(
    context,
    a,
    b,
    x_grid,
    response_sd=response_sd_default,
    Rmax=Rmax,
    num_r=num_r_default,
):
    px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])
    return posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px,
        a=a,
        b=b,
        response_sd=response_sd,
        Rmax=Rmax,
        num_r=num_r,
    )


def decoded_mean_given_x_from_stats(stats):
    x_grid = stats["x_grid"]
    pr_given_x = stats["pr_given_x"]
    xhat_r = stats["posterior_mean"]

    decoded_mean_given_x = np.sum(pr_given_x * xhat_r[None, :], axis=1)
    bias_given_x = decoded_mean_given_x - x_grid
    local_mse_given_x = np.sum(
        pr_given_x * (xhat_r[None, :] - x_grid[:, None]) ** 2,
        axis=1
    )

    return {
        "x_grid": x_grid,
        "decoded_mean_given_x": decoded_mean_given_x,
        "bias_given_x": bias_given_x,
        "local_mse_given_x": local_mse_given_x,
    }


def interp_from_package(r, package, field):
    vals = package[field]
    grid = package["r_grid"]
    return np.interp(r, grid, vals, left=vals[0], right=vals[-1])


def predictive_density(r, package, response_sd=response_sd_default):
    density = np.sum(
        package["px"] * scipy.stats.norm.pdf(
            r,
            loc=package["mean_responses"],
            scale=response_sd
        )
    )
    return max(float(density), 1e-300)
# ---------- one-neuron objectives + multistart ----------

def specialized_mse_objective(params, context, x_grid, response_sd=response_sd_default):
    a, b = params

    if a <= 0:
        return np.inf

    stats = posterior_stats_one_neuron(
        context=context,
        a=a,
        b=b,
        x_grid=x_grid,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def static_mse_objective(params, contexts, x_grid, response_sd=response_sd_default):
    a, b = params

    if a <= 0:
        return np.inf

    px_mix = mixture_prior_pmf(contexts, x_grid)
    stats = posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px_mix,
        a=a,
        b=b,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def context_inference_objective(params, contexts, x_grid, response_sd=response_sd_default):
    """
    Shared inference target objective.
    Minimize average conditional entropy H(C | R) under equal context priors.
    """
    a, b = params

    if a <= 0:
        return np.inf

    mean_responses = logistic(x_grid, a, b, Rmax=Rmax)

    r_min = mean_responses.min() - 4 * response_sd
    r_max = mean_responses.max() + 4 * response_sd
    r_grid = np.linspace(r_min, r_max, num_r_default)

    pr_given_x_density = scipy.stats.norm.pdf(
        r_grid[None, :],
        loc=mean_responses[:, None],
        scale=response_sd
    )
    pr_given_x_density = np.maximum(pr_given_x_density, 1e-300)
    pr_given_x = pr_given_x_density / pr_given_x_density.sum(axis=1, keepdims=True)

    p_r_given_context = {}

    for k, context in contexts.items():
        px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])

        p_r = np.sum(px[:, None] * pr_given_x, axis=0)
        p_r = np.maximum(p_r, 1e-300)
        p_r /= p_r.sum()

        p_r_given_context[k] = p_r

    p_context = {k: 1 / len(contexts) for k in contexts}

    p_r_total = np.zeros_like(r_grid, dtype=float)
    for k in contexts:
        p_r_total += p_context[k] * p_r_given_context[k]

    p_r_total = np.maximum(p_r_total, 1e-300)

    loss = 0.0
    for k in contexts:
        p_c_given_r = (p_context[k] * p_r_given_context[k]) / p_r_total
        p_c_given_r = np.maximum(p_c_given_r, 1e-300)

        loss += -p_context[k] * np.sum(
            p_r_given_context[k] * np.log(p_c_given_r)
        )

    return float(loss)


def fit_multistart(obj, bounds, n_starts=n_starts_default, seed=0, maxiter=maxiter_default):
    rng_local = np.random.default_rng(seed)
    best_res = None

    for _ in range(n_starts):
        x0 = np.array([rng_local.uniform(lo, hi) for lo, hi in bounds])

        res = scipy.optimize.minimize(
            obj,
            x0=x0,
            method="Powell",
            bounds=bounds,
            options={
                "maxiter": maxiter,
                "xtol": 1e-3,
                "ftol": 1e-4,
                "disp": False,
            },
        )

        if (best_res is None) or (res.fun < best_res.fun):
            best_res = res

    return best_res
# ---------- alternative inference target: maximize average classification accuracy ----------

def context_inference_metrics(params, contexts, x_grid, response_sd=response_sd_default, context_weights=None):
    """
    Returns both:
      - conditional_entropy = H(C | R)
      - map_accuracy = sum_r p(r) max_c p(c | r)
    using the same discrete r-grid approximation as the rest of the notebook.
    """
    a, b = params

    if a <= 0:
        return {
            "conditional_entropy": np.inf,
            "map_accuracy": -np.inf,
        }

    mean_responses = logistic(x_grid, a, b, Rmax=Rmax)

    r_min = mean_responses.min() - 4 * response_sd
    r_max = mean_responses.max() + 4 * response_sd
    r_grid = np.linspace(r_min, r_max, num_r_default)

    pr_given_x_density = scipy.stats.norm.pdf(
        r_grid[None, :],
        loc=mean_responses[:, None],
        scale=response_sd
    )
    pr_given_x_density = np.maximum(pr_given_x_density, 1e-300)
    pr_given_x = pr_given_x_density / pr_given_x_density.sum(axis=1, keepdims=True)

    keys = list(contexts.keys())

    if context_weights is None:
        p_context = {k: 1.0 / len(keys) for k in keys}
    else:
        p_context = {k: float(context_weights[k]) for k in keys}
        z = sum(p_context.values())
        p_context = {k: p_context[k] / z for k in keys}

    p_r_given_context = {}
    for k, context in contexts.items():
        px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])
        p_r = np.sum(px[:, None] * pr_given_x, axis=0)
        p_r = np.maximum(p_r, 1e-300)
        p_r /= p_r.sum()
        p_r_given_context[k] = p_r

    p_r_total = np.zeros_like(r_grid, dtype=float)
    for k in keys:
        p_r_total += p_context[k] * p_r_given_context[k]
    p_r_total = np.maximum(p_r_total, 1e-300)
    p_r_total /= p_r_total.sum()

    posterior_mat = []
    conditional_entropy = 0.0

    for k in keys:
        p_c_given_r = (p_context[k] * p_r_given_context[k]) / p_r_total
        p_c_given_r = np.maximum(p_c_given_r, 1e-300)
        posterior_mat.append(p_c_given_r)

        conditional_entropy += -p_context[k] * np.sum(
            p_r_given_context[k] * np.log(p_c_given_r)
        )

    posterior_mat = np.vstack(posterior_mat)   # shape: n_contexts x n_r
    map_accuracy = float(np.sum(p_r_total * np.max(posterior_mat, axis=0)))

    return {
        "conditional_entropy": float(conditional_entropy),
        "map_accuracy": map_accuracy,
    }


def context_accuracy_objective(params, contexts, x_grid, response_sd=response_sd_default, context_weights=None):
    """
    Minimize negative MAP context classification accuracy.
    """
    metrics = context_inference_metrics(
        params=params,
        contexts=contexts,
        x_grid=x_grid,
        response_sd=response_sd,
        context_weights=context_weights,
    )
    return -metrics["map_accuracy"]
# ---------- fit one-neuron codes for both environment types ----------

# mean-switching
x_grid_mean = make_x_grid(contexts_mean)
bounds_mean = [(0.25, 8.0), (x_grid_mean.min(), x_grid_mean.max())]

fit_mean_local_0 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_mean[0], x_grid_mean),
    bounds=bounds_mean,
    n_starts=n_starts_default,
    seed=10,
)

fit_mean_local_1 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_mean[1], x_grid_mean),
    bounds=bounds_mean,
    n_starts=n_starts_default,
    seed=11,
)

fit_mean_static = fit_multistart(
    lambda p: static_mse_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    n_starts=n_starts_default,
    seed=12,
)

fit_mean_infer = fit_multistart(
    lambda p: context_inference_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    n_starts=n_starts_default,
    seed=13,
)

local_params_mean = {
    0: fit_mean_local_0.x,
    1: fit_mean_local_1.x,
}
static_params_mean = fit_mean_static.x
infer_params_mean = fit_mean_infer.x

# variance-switching
x_grid_var = make_x_grid(contexts_var)
bounds_var = [(0.25, 8.0), (x_grid_var.min(), x_grid_var.max())]

fit_var_local_0 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_var[0], x_grid_var),
    bounds=bounds_var,
    n_starts=n_starts_default,
    seed=20,
)

fit_var_local_1 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_var[1], x_grid_var),
    bounds=bounds_var,
    n_starts=n_starts_default,
    seed=21,
)

fit_var_static = fit_multistart(
    lambda p: static_mse_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    n_starts=n_starts_default,
    seed=22,
)

fit_var_infer = fit_multistart(
    lambda p: context_inference_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    n_starts=n_starts_default,
    seed=23,
)

local_params_var = {
    0: fit_var_local_0.x,
    1: fit_var_local_1.x,
}
static_params_var = fit_var_static.x
infer_params_var = fit_var_infer.x

# ---------- 2-context non-local targets ----------
# In the 2-context case, "all other contexts" means just the opposite context,
# so the fully non-local target for context 0 is the local optimum for context 1,
# and vice versa.

nonlocal_target_params_mean = {
    0: np.array(local_params_mean[1], dtype=float),
    1: np.array(local_params_mean[0], dtype=float),
}

nonlocal_target_params_var = {
    0: np.array(local_params_var[1], dtype=float),
    1: np.array(local_params_var[0], dtype=float),
}

print("mean non-local targets:", nonlocal_target_params_mean)
print("variance non-local targets:", nonlocal_target_params_var)

# basic sanity checks
for arr in [
    fit_mean_local_0.x, fit_mean_local_1.x, fit_mean_static.x, fit_mean_infer.x,
    fit_var_local_0.x, fit_var_local_1.x, fit_var_static.x, fit_var_infer.x,
]:
    assert np.all(np.isfinite(arr))

for val in [
    fit_mean_local_0.fun, fit_mean_local_1.fun, fit_mean_static.fun, fit_mean_infer.fun,
    fit_var_local_0.fun, fit_var_local_1.fun, fit_var_static.fun, fit_var_infer.fun,
]:
    assert np.isfinite(val)

print("mean-switching local 0:", fit_mean_local_0.x, "MSE =", fit_mean_local_0.fun)
print("mean-switching local 1:", fit_mean_local_1.x, "MSE =", fit_mean_local_1.fun)
print("mean-switching static :", fit_mean_static.x,  "MSE =", fit_mean_static.fun)
print("mean-switching infer  :", fit_mean_infer.x,   "Inference loss =", fit_mean_infer.fun)
print()
print("variance-switching local 0:", fit_var_local_0.x, "MSE =", fit_var_local_0.fun)
print("variance-switching local 1:", fit_var_local_1.x, "MSE =", fit_var_local_1.fun)
print("variance-switching static :", fit_var_static.x,  "MSE =", fit_var_static.fun)
print("variance-switching infer  :", fit_var_infer.x,   "Inference loss =", fit_var_infer.fun)
# ---------- fit alternative shared inference targets: accuracy-based ----------

fit_mean_infer_acc = fit_multistart(
    lambda p: context_accuracy_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    n_starts=n_starts_default,
    seed=113,
)

fit_var_infer_acc = fit_multistart(
    lambda p: context_accuracy_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    n_starts=n_starts_default,
    seed=123,
)

infer_params_mean_acc = fit_mean_infer_acc.x
infer_params_var_acc = fit_var_infer_acc.x

print("mean accuracy-target params     :", infer_params_mean_acc, "objective =", fit_mean_infer_acc.fun)
print("variance accuracy-target params :", infer_params_var_acc, "objective =", fit_var_infer_acc.fun)
# ---------- compare entropy-target vs accuracy-target ----------

def compare_inference_targets(env_name, contexts, x_grid, infer_params_entropy, infer_params_acc):
    rows = []

    for label, params in [
        ("entropy_target", infer_params_entropy),
        ("accuracy_target", infer_params_acc),
    ]:
        metrics = context_inference_metrics(params, contexts, x_grid)

        rows.append({
            "env": env_name,
            "target": label,
            "a": params[0],
            "b": params[1],
            "H_C_given_R": metrics["conditional_entropy"],
            "map_accuracy": metrics["map_accuracy"],
        })

    return pd.DataFrame(rows)

compare_targets_mean = compare_inference_targets(
    "mean-switching",
    contexts_mean,
    x_grid_mean,
    infer_params_mean,
    infer_params_mean_acc,
)

compare_targets_var = compare_inference_targets(
    "variance-switching",
    contexts_var,
    x_grid_var,
    infer_params_var,
    infer_params_var_acc,
)

print("mean-switching target comparison")
display(compare_targets_mean)

print("variance-switching target comparison")
display(compare_targets_var)
# ---------- plot entropy-target vs accuracy-target ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mean-switching
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[0][0], local_params_mean[0][1]),
    label="local context 0"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[1][0], local_params_mean[1][1]),
    label="local context 1"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean[0], infer_params_mean[1]),
    label="entropy target"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean_acc[0], infer_params_mean_acc[1]),
    label="accuracy target",
    linestyle="--"
)
axes[0].set_title("mean-switching: inference targets")
axes[0].set_xlabel("stimulus x")
axes[0].set_ylabel("response")
axes[0].legend()

# variance-switching
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[0][0], local_params_var[0][1]),
    label="local context 0"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[1][0], local_params_var[1][1]),
    label="local context 1"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var[0], infer_params_var[1]),
    label="entropy target"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var_acc[0], infer_params_var_acc[1]),
    label="accuracy target",
    linestyle="--"
)
axes[1].set_title("variance-switching: inference targets")
axes[1].set_xlabel("stimulus x")
axes[1].set_ylabel("response")
axes[1].legend()

plt.tight_layout()
plt.show()
# ---------- adaptive-family helpers ----------

def blend_params(local_params, target_params, gamma):
    a_local, b_local = local_params
    a_target, b_target = target_params

    return np.array([
        (1 - gamma) * a_local + gamma * a_target,
        (1 - gamma) * b_local + gamma * b_target,
    ])

def target_for_context(target_params, context_key):
    if isinstance(target_params, dict):
        return np.asarray(target_params[context_key], dtype=float)
    return np.asarray(target_params, dtype=float)

def build_local_packages_one_neuron(
    contexts,
    local_params,
    x_grid,
    response_sd=response_sd_default,
):
    packages = {}

    for k, context in contexts.items():
        packages[k] = posterior_stats_one_neuron(
            context=context,
            a=local_params[k][0],
            b=local_params[k][1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return packages


def build_family_packages_one_neuron(
    contexts,
    local_params,
    target_params,
    gamma,
    x_grid,
    response_sd=response_sd_default,
):
    family_params = {}
    family_packages = {}

    for k, context in contexts.items():
        params = blend_params(
            local_params[k],
            target_for_context(target_params, k),
            gamma,
        )
        family_params[k] = params

        family_packages[k] = posterior_stats_one_neuron(
            context=context,
            a=params[0],
            b=params[1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return family_params, family_packages


def build_context_likelihood_packages_one_neuron(
    contexts,
    family_params,
    x_grid,
    response_sd=response_sd_default,
):
    """
    likelihood_packages[used_context][true_context]
    = package for p(r | true_context, encoder_used_on_that_trial)
    """
    likelihood_packages = {}

    for used_k, used_params in family_params.items():
        likelihood_packages[used_k] = {}

        for true_k, context in contexts.items():
            likelihood_packages[used_k][true_k] = posterior_stats_one_neuron(
                context=context,
                a=used_params[0],
                b=used_params[1],
                x_grid=x_grid,
                response_sd=response_sd,
            )

    return likelihood_packages


def build_static_package_one_neuron(
    contexts,
    static_params,
    x_grid,
    response_sd=response_sd_default,
):
    px_mix = mixture_prior_pmf(contexts, x_grid)

    return posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px_mix,
        a=static_params[0],
        b=static_params[1],
        response_sd=response_sd,
    )
# ---------- switching simulation + evaluation ----------

def simulate_context_sequence(n_steps, p_switch=0.01, seed=0):
    rng_local = np.random.default_rng(seed)
    context_seq = np.zeros(n_steps, dtype=int)
    context_seq[0] = rng_local.integers(0, 2)

    for t in range(1, n_steps):
        if rng_local.random() < p_switch:
            context_seq[t] = 1 - context_seq[t - 1]
        else:
            context_seq[t] = context_seq[t - 1]

    return context_seq


def get_mismatch_episode_lengths(df_eval):
    mismatch = df_eval["mismatch"].fillna(False).to_numpy(dtype=bool)

    lengths = []
    current = 0

    for val in mismatch:
        if val:
            current += 1
        elif current > 0:
            lengths.append(current)
            current = 0

    if current > 0:
        lengths.append(current)

    return lengths


def run_strategy_one_neuron(
    strategy_name,
    contexts,
    x_grid,
    local_params,
    static_params,
    target_params=None,
    gamma=0.0,
    p_switch=p_switch_main,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
    seed=0,
):
    rng_local = np.random.default_rng(seed)

    context_seq = simulate_context_sequence(
        n_steps=n_steps,
        p_switch=p_switch,
        seed=seed + 1000,
    )

    local_packages = build_local_packages_one_neuron(
        contexts=contexts,
        local_params=local_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    if target_params is None:
        target_params = static_params

    family_params, family_packages = build_family_packages_one_neuron(
        contexts=contexts,
        local_params=local_params,
        target_params=target_params,
        gamma=gamma,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    context_likelihood_packages = build_context_likelihood_packages_one_neuron(
        contexts=contexts,
        family_params=family_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    static_package = build_static_package_one_neuron(
        contexts=contexts,
        static_params=static_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    transition_matrix = np.array([
        [1 - p_switch, p_switch],
        [p_switch, 1 - p_switch]
    ], dtype=float)

    belief = np.array([0.5, 0.5], dtype=float)

    rows = []

    for t in range(n_steps):
        true_context = int(context_seq[t])
        context_now = contexts[true_context]

        x_t = rng_local.normal(context_now["mu"], context_now["sigma"])

        if strategy_name == "static":
            used_params = static_params
            used_package = static_package
            estimated_context = -1
            belief_pred = belief.copy()

        elif strategy_name == "oracle":
            used_params = local_params[true_context]
            used_package = local_packages[true_context]
            estimated_context = true_context
            belief_pred = belief.copy()

        elif strategy_name == "adaptive":
            belief_pred = transition_matrix.T @ belief
            belief_pred /= belief_pred.sum()

            estimated_context = int(np.argmax(belief_pred))

            used_params = family_params[estimated_context]
            used_package = family_packages[estimated_context]

        else:
            raise ValueError("strategy_name must be 'static', 'oracle', or 'adaptive'")

        r_t = rng_local.normal(
            logistic(x_t, used_params[0], used_params[1]),
            response_sd
        )

        xhat_t = interp_from_package(r_t, used_package, "posterior_mean")
        sq_error_t = (xhat_t - x_t) ** 2

        if strategy_name == "adaptive":
            # true-context likelihoods under the encoder actually used on this trial
            likelihoods = np.array([
                predictive_density(
                    r_t,
                    context_likelihood_packages[estimated_context][0],
                    response_sd=response_sd,
                ),
                predictive_density(
                    r_t,
                    context_likelihood_packages[estimated_context][1],
                    response_sd=response_sd,
                ),
            ])

            belief = likelihoods * belief_pred
            belief_sum = belief.sum()

            if belief_sum <= 0:
                belief = np.array([0.5, 0.5], dtype=float)
            else:
                belief /= belief_sum

        elif strategy_name == "oracle":
            belief = np.eye(2)[true_context]

        rows.append({
            "t": t,
            "true_context": true_context,
            "estimated_context": estimated_context,
            "x": x_t,
            "r": r_t,
            "xhat": xhat_t,
            "sq_error": sq_error_t,
            "mismatch": (estimated_context != true_context) if estimated_context >= 0 else np.nan,
            "belief_context1": belief[1],
        })

    df = pd.DataFrame(rows)
    df_eval = df.iloc[burn_in:].copy()

    out = {
        "strategy": strategy_name,
        "gamma": gamma,
        "p_switch": p_switch,
        "global_mse": df_eval["sq_error"].mean(),
        "df": df,
    }

    if strategy_name == "adaptive":
        matched = df_eval.loc[~df_eval["mismatch"], "sq_error"]
        mismatched = df_eval.loc[df_eval["mismatch"], "sq_error"]
        mismatch_lengths = get_mismatch_episode_lengths(df_eval)

        out["mismatch_rate"] = df_eval["mismatch"].mean()
        out["matched_mse"] = matched.mean() if len(matched) > 0 else np.nan
        out["mismatched_mse"] = mismatched.mean() if len(mismatched) > 0 else np.nan
        out["mean_mismatch_duration"] = np.mean(mismatch_lengths) if len(mismatch_lengths) > 0 else 0.0

    return out


def sweep_gamma_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    target_params,
    family_name,
    gamma_grid,
    p_switch=p_switch_main,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    for gamma in gamma_grid:
        for seed_now in seeds:
            out = run_strategy_one_neuron(
                strategy_name="adaptive",
                contexts=contexts,
                x_grid=x_grid,
                local_params=local_params,
                static_params=static_params,
                target_params=target_params,
                gamma=gamma,
                p_switch=p_switch,
                n_steps=n_steps,
                burn_in=burn_in,
                response_sd=response_sd,
                seed=seed_now,
            )

            rows.append({
                "family": family_name,
                "gamma": gamma,
                "seed": seed_now,
                "global_mse": out["global_mse"],
                "mismatch_rate": out["mismatch_rate"],
                "matched_mse": out["matched_mse"],
                "mismatched_mse": out["mismatched_mse"],
                "mean_mismatch_duration": out["mean_mismatch_duration"],
            })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["family", "gamma"], as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
            mismatch_rate_mean=("mismatch_rate", "mean"),
            matched_mse_mean=("matched_mse", "mean"),
            mismatched_mse_mean=("mismatched_mse", "mean"),
            mean_mismatch_duration_mean=("mean_mismatch_duration", "mean"),
        )
        .sort_values(["family", "gamma"])
        .reset_index(drop=True)
    )

    return df, summary


def evaluate_strategy_set_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    infer_params,
    best_gamma_static,
    best_gamma_infer,
    p_switch=p_switch_main,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    strategy_specs = [
        ("static", None, None),
        ("oracle", None, None),
        ("adaptive_local", static_params, 0.0),
        ("adaptive_best_static", static_params, best_gamma_static),
        ("adaptive_best_infer", infer_params, best_gamma_infer),
    ]

    for seed_now in seeds:
        for label, target_params, gamma in strategy_specs:
            if label == "static":
                out = run_strategy_one_neuron(
                    strategy_name="static",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=static_params,
                    gamma=0.0,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            elif label == "oracle":
                out = run_strategy_one_neuron(
                    strategy_name="oracle",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=static_params,
                    gamma=0.0,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            else:
                out = run_strategy_one_neuron(
                    strategy_name="adaptive",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=target_params,
                    gamma=gamma,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            rows.append({
                "strategy": label,
                "seed": seed_now,
                "global_mse": out["global_mse"],
                "mismatch_rate": out.get("mismatch_rate", np.nan),
                "matched_mse": out.get("matched_mse", np.nan),
                "mismatched_mse": out.get("mismatched_mse", np.nan),
                "mean_mismatch_duration": out.get("mean_mismatch_duration", np.nan),
            })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby("strategy", as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
            mismatch_rate_mean=("mismatch_rate", "mean"),
            matched_mse_mean=("matched_mse", "mean"),
            mismatched_mse_mean=("mismatched_mse", "mean"),
            mean_mismatch_duration_mean=("mean_mismatch_duration", "mean"),
        )
        .sort_values("global_mse_mean")
        .reset_index(drop=True)
    )

    return df, summary


def sweep_switch_prob_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    infer_params,
    best_gamma_static,
    best_gamma_infer,
    p_switch_grid,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    strategy_specs = [
        ("static", None, None),
        ("oracle", None, None),
        ("adaptive_local", static_params, 0.0),
        ("adaptive_best_static", static_params, best_gamma_static),
        ("adaptive_best_infer", infer_params, best_gamma_infer),
    ]

    for p_switch in p_switch_grid:
        for seed_now in seeds:
            for label, target_params, gamma in strategy_specs:
                if label == "static":
                    out = run_strategy_one_neuron(
                        strategy_name="static",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=static_params,
                        gamma=0.0,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                elif label == "oracle":
                    out = run_strategy_one_neuron(
                        strategy_name="oracle",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=static_params,
                        gamma=0.0,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                else:
                    out = run_strategy_one_neuron(
                        strategy_name="adaptive",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=target_params,
                        gamma=gamma,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                rows.append({
                    "p_switch": p_switch,
                    "strategy": label,
                    "seed": seed_now,
                    "global_mse": out["global_mse"],
                })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["p_switch", "strategy"], as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
        )
    )

    return df, summary
# ---------- quick sanity checks before full sweeps ----------

for family_name, target_now in [
        ("shrink_to_static", static_now),
        ("shrink_to_infer", infer_now),
        ("shrink_to_infer_acc", infer_acc_now),
        ("shrink_to_nonlocal", nonlocal_now),
]:
    print(f"\nSanity checks: {env_name}")

    for family_name, target_now in [
        ("shrink_to_static", static_now),
        ("shrink_to_infer", infer_now),
    ]:
        for gamma_now in [0.0, 0.5, 1.0]:
            out = run_strategy_one_neuron(
                strategy_name="adaptive",
                contexts=contexts_now,
                x_grid=x_grid_now,
                local_params=local_now,
                static_params=static_now,
                target_params=target_now,
                gamma=gamma_now,
                n_steps=1000,
                burn_in=100,
                seed=0,
            )

            print(
                f"{family_name} | gamma={gamma_now:.1f} | "
                f"global_mse={out['global_mse']:.4f} | "
                f"mismatch_rate={out['mismatch_rate']:.4f} | "
                f"matched_mse={out['matched_mse']:.4f} | "
                f"mismatched_mse={out['mismatched_mse']:.4f} | "
                f"mean_mismatch_duration={out['mean_mismatch_duration']:.4f}"
            )

            assert np.isfinite(out["global_mse"])
            assert np.isfinite(out["mismatch_rate"])
            assert 0.0 <= out["mismatch_rate"] <= 1.0
            assert np.isfinite(out["matched_mse"]) or np.isnan(out["matched_mse"])
            assert np.isfinite(out["mismatched_mse"]) or np.isnan(out["mismatched_mse"])
            assert np.isfinite(out["mean_mismatch_duration"])

print("\nAll quick sanity checks passed.")
# ---------- gamma sweeps: mean-switching and variance-switching ----------

family_label_map = {
    "shrink_to_static": "toward static target",
    "shrink_to_infer": "toward inference target",
}

# mean-switching
gamma_df_mean_static, gamma_summary_mean_static = sweep_gamma_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=static_params_mean,
    family_name="shrink_to_static",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_df_mean_infer, gamma_summary_mean_infer = sweep_gamma_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=infer_params_mean,
    family_name="shrink_to_infer",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_summary_mean = pd.concat(
    [gamma_summary_mean_static, gamma_summary_mean_infer],
    ignore_index=True
)

best_gamma_mean_static = float(
    gamma_summary_mean_static.loc[
        gamma_summary_mean_static["global_mse_mean"].idxmin(), "gamma"
    ]
)

best_gamma_mean_infer = float(
    gamma_summary_mean_infer.loc[
        gamma_summary_mean_infer["global_mse_mean"].idxmin(), "gamma"
    ]
)

best_row_mean = gamma_summary_mean.loc[
    gamma_summary_mean["global_mse_mean"].idxmin()
]

best_family_mean = best_row_mean["family"]
best_gamma_mean_overall = float(best_row_mean["gamma"])
best_target_params_mean = (
    static_params_mean if best_family_mean == "shrink_to_static" else infer_params_mean
)
best_target_label_mean = family_label_map[best_family_mean]

# variance-switching
gamma_df_var_static, gamma_summary_var_static = sweep_gamma_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=static_params_var,
    family_name="shrink_to_static",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_df_var_infer, gamma_summary_var_infer = sweep_gamma_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=infer_params_var,
    family_name="shrink_to_infer",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_summary_var = pd.concat(
    [gamma_summary_var_static, gamma_summary_var_infer],
    ignore_index=True
)

best_gamma_var_static = float(
    gamma_summary_var_static.loc[
        gamma_summary_var_static["global_mse_mean"].idxmin(), "gamma"
    ]
)

best_gamma_var_infer = float(
    gamma_summary_var_infer.loc[
        gamma_summary_var_infer["global_mse_mean"].idxmin(), "gamma"
    ]
)

best_row_var = gamma_summary_var.loc[
    gamma_summary_var["global_mse_mean"].idxmin()
]

best_family_var = best_row_var["family"]
best_gamma_var_overall = float(best_row_var["gamma"])
best_target_params_var = (
    static_params_var if best_family_var == "shrink_to_static" else infer_params_var
)
best_target_label_var = family_label_map[best_family_var]

print("best gamma (mean, toward static target):", best_gamma_mean_static)
print("best gamma (mean, toward inference target):", best_gamma_mean_infer)
print("overall best mean family:", best_target_label_mean, "| gamma =", best_gamma_mean_overall)
display(gamma_summary_mean)

print("best gamma (variance, toward static target):", best_gamma_var_static)
print("best gamma (variance, toward inference target):", best_gamma_var_infer)
print("overall best variance family:", best_target_label_var, "| gamma =", best_gamma_var_overall)
display(gamma_summary_var)
# ---------- gamma sweeps for accuracy-based inference target ----------

gamma_df_mean_infer_acc, gamma_summary_mean_infer_acc = sweep_gamma_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=infer_params_mean_acc,
    family_name="shrink_to_infer_acc",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_df_var_infer_acc, gamma_summary_var_infer_acc = sweep_gamma_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=infer_params_var_acc,
    family_name="shrink_to_infer_acc",
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

print("mean-switching accuracy-target gamma summary")
display(gamma_summary_mean_infer_acc)

print("variance-switching accuracy-target gamma summary")
display(gamma_summary_var_infer_acc)
# ---------- compare entropy-target vs accuracy-target gamma curves ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(
    gamma_summary_mean_infer["gamma"],
    gamma_summary_mean_infer["global_mse_mean"],
    yerr=gamma_summary_mean_infer["global_mse_std"],
    marker="o",
    capsize=3,
    label="entropy target"
)
axes[0].errorbar(
    gamma_summary_mean_infer_acc["gamma"],
    gamma_summary_mean_infer_acc["global_mse_mean"],
    yerr=gamma_summary_mean_infer_acc["global_mse_std"],
    marker="o",
    capsize=3,
    label="accuracy target"
)
axes[0].set_title("mean-switching: infer-family gamma sweep")
axes[0].set_xlabel("gamma")
axes[0].set_ylabel("global MSE")
axes[0].legend()

axes[1].errorbar(
    gamma_summary_var_infer["gamma"],
    gamma_summary_var_infer["global_mse_mean"],
    yerr=gamma_summary_var_infer["global_mse_std"],
    marker="o",
    capsize=3,
    label="entropy target"
)
axes[1].errorbar(
    gamma_summary_var_infer_acc["gamma"],
    gamma_summary_var_infer_acc["global_mse_mean"],
    yerr=gamma_summary_var_infer_acc["global_mse_std"],
    marker="o",
    capsize=3,
    label="accuracy target"
)
axes[1].set_title("variance-switching: infer-family gamma sweep")
axes[1].set_xlabel("gamma")
axes[1].set_ylabel("global MSE")
axes[1].legend()

plt.tight_layout()
plt.show()
# ---------- main strategy comparison ----------

main_df_mean, main_summary_mean = evaluate_strategy_set_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    best_gamma_static=best_gamma_mean_static,
    best_gamma_infer=best_gamma_mean_infer,
    p_switch=p_switch_main,
)

main_df_var, main_summary_var = evaluate_strategy_set_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    best_gamma_static=best_gamma_var_static,
    best_gamma_infer=best_gamma_var_infer,
    p_switch=p_switch_main,
)

print("main strategy comparison: mean-switching")
display(main_summary_mean)

print("main strategy comparison: variance-switching")
display(main_summary_var)
# ---------- fitted nonlinearities plot ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mean-switching
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[0][0], local_params_mean[0][1]),
    label="local context 0"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[1][0], local_params_mean[1][1]),
    label="local context 1"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, static_params_mean[0], static_params_mean[1]),
    label="static target"
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean[0], infer_params_mean[1]),
    label="inference target"
)
axes[0].set_title("mean-switching: fitted nonlinearities")
axes[0].set_xlabel("stimulus x")
axes[0].set_ylabel("response")
axes[0].legend()

# variance-switching
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[0][0], local_params_var[0][1]),
    label="local context 0"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[1][0], local_params_var[1][1]),
    label="local context 1"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, static_params_var[0], static_params_var[1]),
    label="static target"
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var[0], infer_params_var[1]),
    label="inference target"
)
axes[1].set_title("variance-switching: fitted nonlinearities")
axes[1].set_xlabel("stimulus x")
axes[1].set_ylabel("response")
axes[1].legend()

plt.tight_layout()
plt.show()
# ---------- gamma sweep plots ----------

plot_specs = [
    ("toward static target",   gamma_summary_mean_static,    gamma_summary_var_static),
    ("toward entropy target",  gamma_summary_mean_infer,     gamma_summary_var_infer),
    ("toward accuracy target", gamma_summary_mean_infer_acc, gamma_summary_var_infer_acc),
]

# use gamma=1 of the static-target family as the static-code reference line
mean_static_baseline = gamma_summary_mean_static.loc[
    np.isclose(gamma_summary_mean_static["gamma"], 1.0),
    "global_mse_mean"
].iloc[0]

var_static_baseline = gamma_summary_var_static.loc[
    np.isclose(gamma_summary_var_static["gamma"], 1.0),
    "global_mse_mean"
].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---------------- mean-switching: global MSE ----------------
for label, mean_summary, _ in plot_specs:
    axes[0, 0].errorbar(
        mean_summary["gamma"],
        mean_summary["global_mse_mean"],
        yerr=mean_summary["global_mse_std"],
        marker="o",
        capsize=3,
        label=label,
    )

    best_idx = mean_summary["global_mse_mean"].idxmin()
    axes[0, 0].scatter(
        mean_summary.loc[best_idx, "gamma"],
        mean_summary.loc[best_idx, "global_mse_mean"],
        s=60,
        zorder=5,
    )

axes[0, 0].axhline(
    mean_static_baseline,
    linestyle="--",
    linewidth=1.5,
    label="static-code level",
)
axes[0, 0].set_title("mean-switching: global MSE vs gamma")
axes[0, 0].set_xlabel("gamma (bias toward target)")
axes[0, 0].set_ylabel("global MSE")
axes[0, 0].legend()

# ---------------- variance-switching: global MSE ----------------
for label, _, var_summary in plot_specs:
    axes[0, 1].errorbar(
        var_summary["gamma"],
        var_summary["global_mse_mean"],
        yerr=var_summary["global_mse_std"],
        marker="o",
        capsize=3,
        label=label,
    )

    best_idx = var_summary["global_mse_mean"].idxmin()
    axes[0, 1].scatter(
        var_summary.loc[best_idx, "gamma"],
        var_summary.loc[best_idx, "global_mse_mean"],
        s=60,
        zorder=5,
    )

axes[0, 1].axhline(
    var_static_baseline,
    linestyle="--",
    linewidth=1.5,
    label="static-code level",
)
axes[0, 1].set_title("variance-switching: global MSE vs gamma")
axes[0, 1].set_xlabel("gamma (bias toward target)")
axes[0, 1].set_ylabel("global MSE")
axes[0, 1].legend()

# ---------------- mean-switching: mismatch rate ----------------
for label, mean_summary, _ in plot_specs:
    axes[1, 0].plot(
        mean_summary["gamma"],
        mean_summary["mismatch_rate_mean"],
        marker="o",
        label=label,
    )

axes[1, 0].set_title("mean-switching: mismatch rate vs gamma")
axes[1, 0].set_xlabel("gamma")
axes[1, 0].set_ylabel("mismatch rate")
axes[1, 0].legend()

# ---------------- variance-switching: mismatch rate ----------------
for label, _, var_summary in plot_specs:
    axes[1, 1].plot(
        var_summary["gamma"],
        var_summary["mismatch_rate_mean"],
        marker="o",
        label=label,
    )

axes[1, 1].set_title("variance-switching: mismatch rate vs gamma")
axes[1, 1].set_xlabel("gamma")
axes[1, 1].set_ylabel("mismatch rate")
axes[1, 1].legend()

plt.tight_layout()
plt.show()
# ---------- volatility sweep (p_switch sweep) ----------

p_switch_grid = [0.002, 0.005, 0.01, 0.02, 0.04, 0.08]

switch_df_mean, switch_summary_mean = sweep_switch_prob_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    best_gamma_static=best_gamma_mean_static,
    best_gamma_infer=best_gamma_mean_infer,
    p_switch_grid=p_switch_grid,
)

switch_df_var, switch_summary_var = sweep_switch_prob_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    best_gamma_static=best_gamma_var_static,
    best_gamma_infer=best_gamma_var_infer,
    p_switch_grid=p_switch_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for strategy in switch_summary_mean["strategy"].unique():
    sub = switch_summary_mean[switch_summary_mean["strategy"] == strategy]
    axes[0].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[0].set_title("mean-switching: global MSE vs switch probability")
axes[0].set_xlabel("switch probability")
axes[0].set_ylabel("global MSE")
axes[0].legend()

for strategy in switch_summary_var["strategy"].unique():
    sub = switch_summary_var[switch_summary_var["strategy"] == strategy]
    axes[1].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[1].set_title("variance-switching: global MSE vs switch probability")
axes[1].set_xlabel("switch probability")
axes[1].set_ylabel("global MSE")
axes[1].legend()

plt.tight_layout()
plt.show()
# ---------- mechanistic distortion plots ----------

# choose best overall family per environment
mean_best_params_0 = blend_params(local_params_mean[0], best_target_params_mean, best_gamma_mean_overall)
mean_best_params_1 = blend_params(local_params_mean[1], best_target_params_mean, best_gamma_mean_overall)

var_best_params_0 = blend_params(local_params_var[0], best_target_params_var, best_gamma_var_overall)
var_best_params_1 = blend_params(local_params_var[1], best_target_params_var, best_gamma_var_overall)

# mean-switching stats
mean_local_stats_0 = posterior_stats_one_neuron(contexts_mean[0], *local_params_mean[0], x_grid=x_grid_mean)
mean_local_stats_1 = posterior_stats_one_neuron(contexts_mean[1], *local_params_mean[1], x_grid=x_grid_mean)

mean_best_stats_0 = posterior_stats_one_neuron(contexts_mean[0], *mean_best_params_0, x_grid=x_grid_mean)
mean_best_stats_1 = posterior_stats_one_neuron(contexts_mean[1], *mean_best_params_1, x_grid=x_grid_mean)

mean_local_dec_0 = decoded_mean_given_x_from_stats(mean_local_stats_0)
mean_local_dec_1 = decoded_mean_given_x_from_stats(mean_local_stats_1)

mean_best_dec_0 = decoded_mean_given_x_from_stats(mean_best_stats_0)
mean_best_dec_1 = decoded_mean_given_x_from_stats(mean_best_stats_1)

# variance-switching stats
var_local_stats_0 = posterior_stats_one_neuron(contexts_var[0], *local_params_var[0], x_grid=x_grid_var)
var_local_stats_1 = posterior_stats_one_neuron(contexts_var[1], *local_params_var[1], x_grid=x_grid_var)

var_best_stats_0 = posterior_stats_one_neuron(contexts_var[0], *var_best_params_0, x_grid=x_grid_var)
var_best_stats_1 = posterior_stats_one_neuron(contexts_var[1], *var_best_params_1, x_grid=x_grid_var)

var_local_dec_0 = decoded_mean_given_x_from_stats(var_local_stats_0)
var_local_dec_1 = decoded_mean_given_x_from_stats(var_local_stats_1)

var_best_dec_0 = decoded_mean_given_x_from_stats(var_best_stats_0)
var_best_dec_1 = decoded_mean_given_x_from_stats(var_best_stats_1)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# mean-switching decoded estimate
axes[0, 0].plot(mean_local_dec_0["x_grid"], mean_local_dec_0["x_grid"], label="identity")
axes[0, 0].plot(mean_local_dec_0["x_grid"], mean_local_dec_0["decoded_mean_given_x"], label="local, context 0")
axes[0, 0].plot(mean_best_dec_0["x_grid"], mean_best_dec_0["decoded_mean_given_x"], label=f"best family, context 0")
axes[0, 0].plot(mean_local_dec_1["x_grid"], mean_local_dec_1["decoded_mean_given_x"], label="local, context 1")
axes[0, 0].plot(mean_best_dec_1["x_grid"], mean_best_dec_1["decoded_mean_given_x"], label=f"best family, context 1")
axes[0, 0].set_title(f"mean-switching: decoded estimate vs true x\nbest = {best_target_label_mean}, gamma={best_gamma_mean_overall:.2f}")
axes[0, 0].set_xlabel("true stimulus x")
axes[0, 0].set_ylabel("E[xhat(R) | x]")
axes[0, 0].legend()

# mean-switching local MSE
axes[0, 1].plot(mean_local_dec_0["x_grid"], mean_local_dec_0["local_mse_given_x"], label="local, context 0")
axes[0, 1].plot(mean_best_dec_0["x_grid"], mean_best_dec_0["local_mse_given_x"], label="best family, context 0")
axes[0, 1].plot(mean_local_dec_1["x_grid"], mean_local_dec_1["local_mse_given_x"], label="local, context 1")
axes[0, 1].plot(mean_best_dec_1["x_grid"], mean_best_dec_1["local_mse_given_x"], label="best family, context 1")
axes[0, 1].set_title("mean-switching: local MSE vs x")
axes[0, 1].set_xlabel("true stimulus x")
axes[0, 1].set_ylabel("E[(xhat(R)-x)^2 | x]")
axes[0, 1].legend()

# variance-switching decoded estimate
axes[1, 0].plot(var_local_dec_0["x_grid"], var_local_dec_0["x_grid"], label="identity")
axes[1, 0].plot(var_local_dec_0["x_grid"], var_local_dec_0["decoded_mean_given_x"], label="local, context 0")
axes[1, 0].plot(var_best_dec_0["x_grid"], var_best_dec_0["decoded_mean_given_x"], label="best family, context 0")
axes[1, 0].plot(var_local_dec_1["x_grid"], var_local_dec_1["decoded_mean_given_x"], label="local, context 1")
axes[1, 0].plot(var_best_dec_1["x_grid"], var_best_dec_1["decoded_mean_given_x"], label="best family, context 1")
axes[1, 0].set_title(f"variance-switching: decoded estimate vs true x\nbest = {best_target_label_var}, gamma={best_gamma_var_overall:.2f}")
axes[1, 0].set_xlabel("true stimulus x")
axes[1, 0].set_ylabel("E[xhat(R) | x]")
axes[1, 0].legend()

# variance-switching local MSE
axes[1, 1].plot(var_local_dec_0["x_grid"], var_local_dec_0["local_mse_given_x"], label="local, context 0")
axes[1, 1].plot(var_best_dec_0["x_grid"], var_best_dec_0["local_mse_given_x"], label="best family, context 0")
axes[1, 1].plot(var_local_dec_1["x_grid"], var_local_dec_1["local_mse_given_x"], label="local, context 1")
axes[1, 1].plot(var_best_dec_1["x_grid"], var_best_dec_1["local_mse_given_x"], label="best family, context 1")
axes[1, 1].set_title("variance-switching: local MSE vs x")
axes[1, 1].set_xlabel("true stimulus x")
axes[1, 1].set_ylabel("E[(xhat(R)-x)^2 | x]")
axes[1, 1].legend()

plt.tight_layout()
plt.show()
# ---------- representative adaptive traces ----------

trace_mean_local = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=static_params_mean,
    gamma=0.0,
    p_switch=p_switch_main,
    seed=123,
)

trace_mean_best = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=best_target_params_mean,
    gamma=best_gamma_mean_overall,
    p_switch=p_switch_main,
    seed=123,
)

trace_var_local = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=static_params_var,
    gamma=0.0,
    p_switch=p_switch_main,
    seed=123,
)

trace_var_best = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=best_target_params_var,
    gamma=best_gamma_var_overall,
    p_switch=p_switch_main,
    seed=123,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

show_slice = slice(500, 900)

axes[0, 0].plot(
    trace_mean_local["df"].iloc[show_slice]["t"],
    trace_mean_local["df"].iloc[show_slice]["true_context"],
    label="true context"
)
axes[0, 0].plot(
    trace_mean_local["df"].iloc[show_slice]["t"],
    trace_mean_local["df"].iloc[show_slice]["estimated_context"],
    label="estimated context"
)
axes[0, 0].set_title("mean-switching: naive local adaptive")
axes[0, 0].set_ylabel("context")
axes[0, 0].legend()

axes[0, 1].plot(
    trace_mean_best["df"].iloc[show_slice]["t"],
    trace_mean_best["df"].iloc[show_slice]["true_context"],
    label="true context"
)
axes[0, 1].plot(
    trace_mean_best["df"].iloc[show_slice]["t"],
    trace_mean_best["df"].iloc[show_slice]["estimated_context"],
    label="estimated context"
)
axes[0, 1].set_title(f"mean-switching: best adaptive\n{best_target_label_mean}, gamma={best_gamma_mean_overall:.2f}")
axes[0, 1].set_ylabel("context")
axes[0, 1].legend()

axes[1, 0].plot(
    trace_var_local["df"].iloc[show_slice]["t"],
    trace_var_local["df"].iloc[show_slice]["true_context"],
    label="true context"
)
axes[1, 0].plot(
    trace_var_local["df"].iloc[show_slice]["t"],
    trace_var_local["df"].iloc[show_slice]["estimated_context"],
    label="estimated context"
)
axes[1, 0].set_title("variance-switching: naive local adaptive")
axes[1, 0].set_xlabel("time step")
axes[1, 0].set_ylabel("context")
axes[1, 0].legend()

axes[1, 1].plot(
    trace_var_best["df"].iloc[show_slice]["t"],
    trace_var_best["df"].iloc[show_slice]["true_context"],
    label="true context"
)
axes[1, 1].plot(
    trace_var_best["df"].iloc[show_slice]["t"],
    trace_var_best["df"].iloc[show_slice]["estimated_context"],
    label="estimated context"
)
axes[1, 1].set_title(f"variance-switching: best adaptive\n{best_target_label_var}, gamma={best_gamma_var_overall:.2f}")
axes[1, 1].set_xlabel("time step")
axes[1, 1].set_ylabel("context")
axes[1, 1].legend()

plt.tight_layout()
plt.show()
# ---------- optional 2-neuron helpers ----------

def posterior_stats_from_prior_two_neurons(
    x_grid,
    px,
    a1,
    b1,
    a2,
    b2,
    response_sd=response_sd_default,
    Rmax=Rmax,
    num_r=81,
):
    mu1 = logistic(x_grid, a1, b1, Rmax=Rmax)
    mu2 = logistic(x_grid, a2, b2, Rmax=Rmax)

    r1_grid = np.linspace(mu1.min() - 4 * response_sd, mu1.max() + 4 * response_sd, num_r)
    r2_grid = np.linspace(mu2.min() - 4 * response_sd, mu2.max() + 4 * response_sd, num_r)

    pr1_given_x = scipy.stats.norm.pdf(
        r1_grid[None, :],
        loc=mu1[:, None],
        scale=response_sd
    )
    pr2_given_x = scipy.stats.norm.pdf(
        r2_grid[None, :],
        loc=mu2[:, None],
        scale=response_sd
    )

    pr1_given_x = np.maximum(pr1_given_x, 1e-300)
    pr2_given_x = np.maximum(pr2_given_x, 1e-300)

    pr1_given_x /= pr1_given_x.sum(axis=1, keepdims=True)
    pr2_given_x /= pr2_given_x.sum(axis=1, keepdims=True)

    pr_joint_given_x = pr1_given_x[:, :, None] * pr2_given_x[:, None, :]

    p_r1_r2 = np.sum(px[:, None, None] * pr_joint_given_x, axis=0)
    p_r1_r2 = np.maximum(p_r1_r2, 1e-300)
    p_r1_r2 /= p_r1_r2.sum()

    posterior = (pr_joint_given_x * px[:, None, None]) / p_r1_r2    [None, :, :]
    posterior = np.maximum(posterior, 1e-300)
    posterior /= posterior.sum(axis=0, keepdims=True)

    posterior_mean = np.sum(posterior * x_grid[:, None, None], axis=0)

    bayes_mse = np.sum(
        px[:, None, None] *
        pr_joint_given_x *
        (posterior_mean[None, :, :] - x_grid[:, None, None]) ** 2
    )

    return {
        "x_grid": x_grid,
        "px": px,
        "r1_grid": r1_grid,
        "r2_grid": r2_grid,
        "posterior_mean": posterior_mean,
        "bayes_mse": bayes_mse,
    }


def specialized_mse_objective_two(params, context, x_grid, response_sd=response_sd_default):
    a1, b1, a2, b2 = params

    if (a1 <= 0) or (a2 <= 0):
        return np.inf

    px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])

    stats = posterior_stats_from_prior_two_neurons(
        x_grid=x_grid,
        px=px,
        a1=a1,
        b1=b1,
        a2=a2,
        b2=b2,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def static_mse_objective_two(params, contexts, x_grid, response_sd=response_sd_default):
    a1, b1, a2, b2 = params

    if (a1 <= 0) or (a2 <= 0):
        return np.inf

    px_mix = mixture_prior_pmf(contexts, x_grid)

    stats = posterior_stats_from_prior_two_neurons(
        x_grid=x_grid,
        px=px_mix,
        a1=a1,
        b1=b1,
        a2=a2,
        b2=b2,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def fit_multistart_two(obj, bounds, n_starts=8, seed=0, maxiter=120):
    rng_local = np.random.default_rng(seed)
    best_res = None

    for _ in range(n_starts):
        x0 = np.array([rng_local.uniform(lo, hi) for lo, hi in bounds])

        res = scipy.optimize.minimize(
            obj,
            x0=x0,
            method="Powell",
            bounds=bounds,
            options={
                "maxiter": maxiter,
                "xtol": 1e-3,
                "ftol": 1e-4,
                "disp": False,
            },
        )

        if (best_res is None) or (res.fun < best_res.fun):
            best_res = res

    return best_res
# ---------- optional 2-neuron fit: do one representative case only ----------

bounds_mean_two = [
    (0.25, 8.0), (x_grid_mean.min(), x_grid_mean.max()),
    (0.25, 8.0), (x_grid_mean.min(), x_grid_mean.max()),
]

fit_mean_local_0_two = fit_multistart_two(
    lambda p: specialized_mse_objective_two(p, contexts_mean[0], x_grid_mean),
    bounds=bounds_mean_two,
    n_starts=8,
    seed=300,
)

fit_mean_local_1_two = fit_multistart_two(
    lambda p: specialized_mse_objective_two(p, contexts_mean[1], x_grid_mean),
    bounds=bounds_mean_two,
    n_starts=8,
    seed=301,
)

fit_mean_static_two = fit_multistart_two(
    lambda p: static_mse_objective_two(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean_two,
    n_starts=8,
    seed=302,
)

print("2-neuron mean local 0:", fit_mean_local_0_two.x, "MSE =", fit_mean_local_0_two.fun)
print("2-neuron mean local 1:", fit_mean_local_1_two.x, "MSE =", fit_mean_local_1_two.fun)
print("2-neuron mean static :", fit_mean_static_two.x,  "MSE =", fit_mean_static_two.fun)
# ---------- optional 2-neuron plots ----------

a1, b1, a2, b2 = fit_mean_static_two.x

plt.figure(figsize=(10, 6))

plt.plot(
    x_grid_mean,
    scipy.stats.norm.pdf(x_grid_mean, contexts_mean[0]["mu"], contexts_mean[0]["sigma"]),
    label="context 0 prior"
)
plt.plot(
    x_grid_mean,
    scipy.stats.norm.pdf(x_grid_mean, contexts_mean[1]["mu"], contexts_mean[1]["sigma"]),
    label="context 1 prior"
)

plt.plot(x_grid_mean, logistic(x_grid_mean, a1, b1), "--", label="2-neuron static: neuron 1")
plt.plot(x_grid_mean, logistic(x_grid_mean, a2, b2), "--", label="2-neuron static: neuron 2")

plt.plot(
    x_grid_mean,
    logistic_derivative(x_grid_mean, a1, b1) + logistic_derivative(x_grid_mean, a2, b2),
    linewidth=3,
    label="total sensitivity"
)

plt.xlabel("stimulus x")
plt.ylabel("density / response / sensitivity")
plt.title("optional 2-neuron extension: mean-switching static code")
plt.legend()
plt.tight_layout()
plt.show()


rows_two_compare = [
    {
        "model": "1-neuron static",
        "MSE": fit_mean_static.fun,
    },
    {
        "model": "2-neuron static",
        "MSE": fit_mean_static_two.fun,
    },
    {
        "model": "1-neuron local 0",
        "MSE": fit_mean_local_0.fun,
    },
    {
        "model": "2-neuron local 0",
        "MSE": fit_mean_local_0_two.fun,
    },
    {
        "model": "1-neuron local 1",
        "MSE": fit_mean_local_1.fun,
    },
    {
        "model": "2-neuron local 1",
        "MSE": fit_mean_local_1_two.fun,
    },
]

df_two_compare = pd.DataFrame(rows_two_compare)
display(df_two_compare)

In [ ]:
# ---------- contexts ----------

contexts_mean = {
    0: {"name": "low-mean",  "mu": -2.0, "sigma": 1.0},
    1: {"name": "high-mean", "mu":  2.0, "sigma": 1.0},
}

contexts_var = {
    0: {"name": "low-var",  "mu": 0.0, "sigma": 0.7},
    1: {"name": "high-var", "mu": 0.0, "sigma": 1.8},
}

In [ ]:
# ---------- core one-neuron helpers ----------

def logistic(x, a, b, Rmax=Rmax):
    return Rmax / (1 + np.exp(-a * (x - b)))


def logistic_derivative(x, a, b, Rmax=Rmax):
    y = logistic(x, a, b, Rmax=Rmax)
    return a * y * (1 - y / Rmax)


def make_x_grid(contexts, n_std=5, num_x=num_x_default):
    mus = [c["mu"] for c in contexts.values()]
    sigmas = [c["sigma"] for c in contexts.values()]

    x_min = min(mu - n_std * sigma for mu, sigma in zip(mus, sigmas))
    x_max = max(mu + n_std * sigma for mu, sigma in zip(mus, sigmas))

    pad = 0.5
    return np.linspace(x_min - pad, x_max + pad, num_x)


def normal_pmf_on_grid(x_grid, mu, sigma):
    px = scipy.stats.norm.pdf(x_grid, loc=mu, scale=sigma)
    px = np.maximum(px, 1e-300)
    px /= px.sum()
    return px


def mixture_prior_pmf(contexts, x_grid, weights=None):
    keys = list(contexts.keys())

    if weights is None:
        weights = {k: 1.0 / len(keys) for k in keys}

    px = np.zeros_like(x_grid, dtype=float)

    for k in keys:
        px += weights[k] * normal_pmf_on_grid(
            x_grid,
            contexts[k]["mu"],
            contexts[k]["sigma"],
        )

    px = np.maximum(px, 1e-300)
    px /= px.sum()
    return px


def posterior_stats_from_prior_one_neuron(
    x_grid,
    px,
    a,
    b,
    response_sd=response_sd_default,
    Rmax=Rmax,
    num_r=num_r_default,
):
    mean_responses = logistic(x_grid, a, b, Rmax=Rmax)

    r_min = mean_responses.min() - 4 * response_sd
    r_max = mean_responses.max() + 4 * response_sd
    r_grid = np.linspace(r_min, r_max, num_r)

    pr_given_x_density = scipy.stats.norm.pdf(
        r_grid[None, :],
        loc=mean_responses[:, None],
        scale=response_sd,
    )
    pr_given_x_density = np.maximum(pr_given_x_density, 1e-300)

    pr_given_x = pr_given_x_density / pr_given_x_density.sum(axis=1, keepdims=True)

    p_r = np.sum(px[:, None] * pr_given_x, axis=0)
    p_r = np.maximum(p_r, 1e-300)
    p_r /= p_r.sum()

    posterior = (pr_given_x * px[:, None]) / p_r[None, :]
    posterior = np.maximum(posterior, 1e-300)
    posterior /= posterior.sum(axis=0, keepdims=True)

    posterior_mean = np.sum(posterior * x_grid[:, None], axis=0)

    bayes_mse = np.sum(
        px[:, None] * pr_given_x * (posterior_mean[None, :] - x_grid[:, None]) ** 2
    )

    return {
        "x_grid": x_grid,
        "px": px,
        "a": a,
        "b": b,
        "mean_responses": mean_responses,
        "r_grid": r_grid,
        "pr_given_x_density": pr_given_x_density,
        "pr_given_x": pr_given_x,
        "p_r": p_r,
        "posterior": posterior,
        "posterior_mean": posterior_mean,
        "bayes_mse": bayes_mse,
    }


def posterior_stats_one_neuron(
    context,
    a,
    b,
    x_grid,
    response_sd=response_sd_default,
    Rmax=Rmax,
    num_r=num_r_default,
):
    px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])
    return posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px,
        a=a,
        b=b,
        response_sd=response_sd,
        Rmax=Rmax,
        num_r=num_r,
    )


def decoded_mean_given_x_from_stats(stats):
    x_grid = stats["x_grid"]
    pr_given_x = stats["pr_given_x"]
    xhat_r = stats["posterior_mean"]

    decoded_mean_given_x = np.sum(pr_given_x * xhat_r[None, :], axis=1)
    bias_given_x = decoded_mean_given_x - x_grid
    local_mse_given_x = np.sum(
        pr_given_x * (xhat_r[None, :] - x_grid[:, None]) ** 2,
        axis=1,
    )

    return {
        "x_grid": x_grid,
        "decoded_mean_given_x": decoded_mean_given_x,
        "bias_given_x": bias_given_x,
        "local_mse_given_x": local_mse_given_x,
    }


def interp_from_package(r, package, field):
    vals = package[field]
    grid = package["r_grid"]
    return np.interp(r, grid, vals, left=vals[0], right=vals[-1])


def predictive_density(r, package, response_sd=response_sd_default):
    density = np.sum(
        package["px"] * scipy.stats.norm.pdf(
            r,
            loc=package["mean_responses"],
            scale=response_sd,
        )
    )
    return max(float(density), 1e-300)

In [ ]:
# ---------- objectives + multistart ----------

def specialized_mse_objective(params, context, x_grid, response_sd=response_sd_default):
    a, b = params
    if a <= 0:
        return np.inf

    stats = posterior_stats_one_neuron(
        context=context,
        a=a,
        b=b,
        x_grid=x_grid,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def static_mse_objective(params, contexts, x_grid, response_sd=response_sd_default):
    a, b = params
    if a <= 0:
        return np.inf

    px_mix = mixture_prior_pmf(contexts, x_grid)
    stats = posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px_mix,
        a=a,
        b=b,
        response_sd=response_sd,
    )
    return stats["bayes_mse"]


def context_inference_metrics(
    params,
    contexts,
    x_grid,
    response_sd=response_sd_default,
    context_weights=None,
):
    """
    Returns:
      - conditional_entropy = H(C | R)
      - map_accuracy = sum_r p(r) max_c p(c | r)
    """
    a, b = params
    if a <= 0:
        return {
            "conditional_entropy": np.inf,
            "map_accuracy": -np.inf,
        }

    mean_responses = logistic(x_grid, a, b, Rmax=Rmax)

    r_min = mean_responses.min() - 4 * response_sd
    r_max = mean_responses.max() + 4 * response_sd
    r_grid = np.linspace(r_min, r_max, num_r_default)

    pr_given_x_density = scipy.stats.norm.pdf(
        r_grid[None, :],
        loc=mean_responses[:, None],
        scale=response_sd,
    )
    pr_given_x_density = np.maximum(pr_given_x_density, 1e-300)
    pr_given_x = pr_given_x_density / pr_given_x_density.sum(axis=1, keepdims=True)

    keys = list(contexts.keys())

    if context_weights is None:
        p_context = {k: 1.0 / len(keys) for k in keys}
    else:
        p_context = {k: float(context_weights[k]) for k in keys}
        z = sum(p_context.values())
        p_context = {k: p_context[k] / z for k in keys}

    p_r_given_context = {}
    for k, context in contexts.items():
        px = normal_pmf_on_grid(x_grid, context["mu"], context["sigma"])
        p_r = np.sum(px[:, None] * pr_given_x, axis=0)
        p_r = np.maximum(p_r, 1e-300)
        p_r /= p_r.sum()
        p_r_given_context[k] = p_r

    p_r_total = np.zeros_like(r_grid, dtype=float)
    for k in keys:
        p_r_total += p_context[k] * p_r_given_context[k]
    p_r_total = np.maximum(p_r_total, 1e-300)
    p_r_total /= p_r_total.sum()

    posterior_mat = []
    conditional_entropy = 0.0

    for k in keys:
        p_c_given_r = (p_context[k] * p_r_given_context[k]) / p_r_total
        p_c_given_r = np.maximum(p_c_given_r, 1e-300)
        posterior_mat.append(p_c_given_r)

        conditional_entropy += -p_context[k] * np.sum(
            p_r_given_context[k] * np.log(p_c_given_r)
        )

    posterior_mat = np.vstack(posterior_mat)
    map_accuracy = float(np.sum(p_r_total * np.max(posterior_mat, axis=0)))

    return {
        "conditional_entropy": float(conditional_entropy),
        "map_accuracy": map_accuracy,
    }


def context_inference_objective(params, contexts, x_grid, response_sd=response_sd_default):
    metrics = context_inference_metrics(
        params=params,
        contexts=contexts,
        x_grid=x_grid,
        response_sd=response_sd,
    )
    return metrics["conditional_entropy"]


def context_accuracy_objective(
    params,
    contexts,
    x_grid,
    response_sd=response_sd_default,
    context_weights=None,
):
    metrics = context_inference_metrics(
        params=params,
        contexts=contexts,
        x_grid=x_grid,
        response_sd=response_sd,
        context_weights=context_weights,
    )
    return -metrics["map_accuracy"]


def fit_multistart(
    obj,
    bounds,
    n_starts=n_starts_default,
    seed=0,
    maxiter=maxiter_default,
):
    rng_local = np.random.default_rng(seed)
    best_res = None

    for _ in range(n_starts):
        x0 = np.array([rng_local.uniform(lo, hi) for lo, hi in bounds])

        res = scipy.optimize.minimize(
            obj,
            x0=x0,
            method="Powell",
            bounds=bounds,
            options={
                "maxiter": maxiter,
                "xtol": 1e-3,
                "ftol": 1e-4,
                "disp": False,
            },
        )

        if (best_res is None) or (res.fun < best_res.fun):
            best_res = res

    return best_res

In [ ]:
# ---------- fit one-neuron codes ----------

# mean-switching
x_grid_mean = make_x_grid(contexts_mean)
bounds_mean = [(0.25, 8.0), (x_grid_mean.min(), x_grid_mean.max())]

fit_mean_local_0 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_mean[0], x_grid_mean),
    bounds=bounds_mean,
    seed=10,
)
fit_mean_local_1 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_mean[1], x_grid_mean),
    bounds=bounds_mean,
    seed=11,
)
fit_mean_static = fit_multistart(
    lambda p: static_mse_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    seed=12,
)
fit_mean_infer = fit_multistart(
    lambda p: context_inference_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    seed=13,
)
fit_mean_infer_acc = fit_multistart(
    lambda p: context_accuracy_objective(p, contexts_mean, x_grid_mean),
    bounds=bounds_mean,
    seed=113,
)

local_params_mean = {
    0: fit_mean_local_0.x,
    1: fit_mean_local_1.x,
}
static_params_mean = fit_mean_static.x
infer_params_mean = fit_mean_infer.x
infer_params_mean_acc = fit_mean_infer_acc.x

# simplified 2-context non-local targets:
# target for one context = local optimum of the opposite context
nonlocal_target_params_mean = {
    0: np.array(local_params_mean[1], dtype=float),
    1: np.array(local_params_mean[0], dtype=float),
}

# variance-switching
x_grid_var = make_x_grid(contexts_var)
bounds_var = [(0.25, 8.0), (x_grid_var.min(), x_grid_var.max())]

fit_var_local_0 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_var[0], x_grid_var),
    bounds=bounds_var,
    seed=20,
)
fit_var_local_1 = fit_multistart(
    lambda p: specialized_mse_objective(p, contexts_var[1], x_grid_var),
    bounds=bounds_var,
    seed=21,
)
fit_var_static = fit_multistart(
    lambda p: static_mse_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    seed=22,
)
fit_var_infer = fit_multistart(
    lambda p: context_inference_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    seed=23,
)
fit_var_infer_acc = fit_multistart(
    lambda p: context_accuracy_objective(p, contexts_var, x_grid_var),
    bounds=bounds_var,
    seed=123,
)

local_params_var = {
    0: fit_var_local_0.x,
    1: fit_var_local_1.x,
}
static_params_var = fit_var_static.x
infer_params_var = fit_var_infer.x
infer_params_var_acc = fit_var_infer_acc.x

nonlocal_target_params_var = {
    0: np.array(local_params_var[1], dtype=float),
    1: np.array(local_params_var[0], dtype=float),
}

# sanity checks
for arr in [
    fit_mean_local_0.x, fit_mean_local_1.x, fit_mean_static.x, fit_mean_infer.x, fit_mean_infer_acc.x,
    fit_var_local_0.x, fit_var_local_1.x, fit_var_static.x, fit_var_infer.x, fit_var_infer_acc.x,
]:
    assert np.all(np.isfinite(arr))

for val in [
    fit_mean_local_0.fun, fit_mean_local_1.fun, fit_mean_static.fun, fit_mean_infer.fun, fit_mean_infer_acc.fun,
    fit_var_local_0.fun, fit_var_local_1.fun, fit_var_static.fun, fit_var_infer.fun, fit_var_infer_acc.fun,
]:
    assert np.isfinite(val)

print("mean-switching")
print("local 0       :", fit_mean_local_0.x, "MSE =", fit_mean_local_0.fun)
print("local 1       :", fit_mean_local_1.x, "MSE =", fit_mean_local_1.fun)
print("static        :", fit_mean_static.x,  "MSE =", fit_mean_static.fun)
print("entropy target:", fit_mean_infer.x,   "loss =", fit_mean_infer.fun)
print("accuracy target:", fit_mean_infer_acc.x, "loss =", fit_mean_infer_acc.fun)
print("non-local targets:", nonlocal_target_params_mean)
print()

print("variance-switching")
print("local 0       :", fit_var_local_0.x, "MSE =", fit_var_local_0.fun)
print("local 1       :", fit_var_local_1.x, "MSE =", fit_var_local_1.fun)
print("static        :", fit_var_static.x,  "MSE =", fit_var_static.fun)
print("entropy target:", fit_var_infer.x,   "loss =", fit_var_infer.fun)
print("accuracy target:", fit_var_infer_acc.x, "loss =", fit_var_infer_acc.fun)
print("non-local targets:", nonlocal_target_params_var)

In [ ]:
# ---------- compare entropy-target vs accuracy-target ----------

def compare_inference_targets(env_name, contexts, x_grid, infer_params_entropy, infer_params_acc):
    rows = []

    for label, params in [
        ("entropy_target", infer_params_entropy),
        ("accuracy_target", infer_params_acc),
    ]:
        metrics = context_inference_metrics(params, contexts, x_grid)

        rows.append({
            "env": env_name,
            "target": label,
            "a": params[0],
            "b": params[1],
            "H_C_given_R": metrics["conditional_entropy"],
            "map_accuracy": metrics["map_accuracy"],
        })

    return pd.DataFrame(rows)


compare_targets_mean = compare_inference_targets(
    "mean-switching",
    contexts_mean,
    x_grid_mean,
    infer_params_mean,
    infer_params_mean_acc,
)

compare_targets_var = compare_inference_targets(
    "variance-switching",
    contexts_var,
    x_grid_var,
    infer_params_var,
    infer_params_var_acc,
)

print("mean-switching target comparison")
display(compare_targets_mean)

print("variance-switching target comparison")
display(compare_targets_var)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mean-switching
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[0][0], local_params_mean[0][1]),
    label="local context 0",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[1][0], local_params_mean[1][1]),
    label="local context 1",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean[0], infer_params_mean[1]),
    label="entropy target",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean_acc[0], infer_params_mean_acc[1]),
    linestyle="--",
    label="accuracy target",
)
axes[0].set_title("mean-switching: inference targets")
axes[0].set_xlabel("stimulus x")
axes[0].set_ylabel("response")
axes[0].legend()

# variance-switching
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[0][0], local_params_var[0][1]),
    label="local context 0",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[1][0], local_params_var[1][1]),
    label="local context 1",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var[0], infer_params_var[1]),
    label="entropy target",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var_acc[0], infer_params_var_acc[1]),
    linestyle="--",
    label="accuracy target",
)
axes[1].set_title("variance-switching: inference targets")
axes[1].set_xlabel("stimulus x")
axes[1].set_ylabel("response")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- adaptive-family helpers ----------

def blend_params(local_params, target_params, gamma):
    a_local, b_local = local_params
    a_target, b_target = target_params

    return np.array([
        (1 - gamma) * a_local + gamma * a_target,
        (1 - gamma) * b_local + gamma * b_target,
    ])


def target_for_context(target_params, context_key):
    if isinstance(target_params, dict):
        return np.asarray(target_params[context_key], dtype=float)
    return np.asarray(target_params, dtype=float)


def build_local_packages_one_neuron(
    contexts,
    local_params,
    x_grid,
    response_sd=response_sd_default,
):
    packages = {}

    for k, context in contexts.items():
        packages[k] = posterior_stats_one_neuron(
            context=context,
            a=local_params[k][0],
            b=local_params[k][1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return packages


def build_family_packages_one_neuron(
    contexts,
    local_params,
    target_params,
    gamma,
    x_grid,
    response_sd=response_sd_default,
):
    family_params = {}
    family_packages = {}

    for k, context in contexts.items():
        params = blend_params(
            local_params[k],
            target_for_context(target_params, k),
            gamma,
        )
        family_params[k] = params

        family_packages[k] = posterior_stats_one_neuron(
            context=context,
            a=params[0],
            b=params[1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return family_params, family_packages


def build_context_likelihood_packages_one_neuron(
    contexts,
    family_params,
    x_grid,
    response_sd=response_sd_default,
):
    """
    likelihood_packages[used_context][true_context]
    = package for p(r | true_context, encoder_used_on_that_trial)
    """
    likelihood_packages = {}

    for used_k, used_params in family_params.items():
        likelihood_packages[used_k] = {}

        for true_k, context in contexts.items():
            likelihood_packages[used_k][true_k] = posterior_stats_one_neuron(
                context=context,
                a=used_params[0],
                b=used_params[1],
                x_grid=x_grid,
                response_sd=response_sd,
            )

    return likelihood_packages


def build_static_package_one_neuron(
    contexts,
    static_params,
    x_grid,
    response_sd=response_sd_default,
):
    px_mix = mixture_prior_pmf(contexts, x_grid)

    return posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px_mix,
        a=static_params[0],
        b=static_params[1],
        response_sd=response_sd,
    )

In [ ]:
# ---------- adaptive-family helpers ----------

def blend_params(local_params, target_params, gamma):
    a_local, b_local = local_params
    a_target, b_target = target_params

    return np.array([
        (1 - gamma) * a_local + gamma * a_target,
        (1 - gamma) * b_local + gamma * b_target,
    ])


def target_for_context(target_params, context_key):
    if isinstance(target_params, dict):
        return np.asarray(target_params[context_key], dtype=float)
    return np.asarray(target_params, dtype=float)


def build_local_packages_one_neuron(
    contexts,
    local_params,
    x_grid,
    response_sd=response_sd_default,
):
    packages = {}

    for k, context in contexts.items():
        packages[k] = posterior_stats_one_neuron(
            context=context,
            a=local_params[k][0],
            b=local_params[k][1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return packages


def build_family_packages_one_neuron(
    contexts,
    local_params,
    target_params,
    gamma,
    x_grid,
    response_sd=response_sd_default,
):
    family_params = {}
    family_packages = {}

    for k, context in contexts.items():
        params = blend_params(
            local_params[k],
            target_for_context(target_params, k),
            gamma,
        )
        family_params[k] = params

        family_packages[k] = posterior_stats_one_neuron(
            context=context,
            a=params[0],
            b=params[1],
            x_grid=x_grid,
            response_sd=response_sd,
        )

    return family_params, family_packages


def build_context_likelihood_packages_one_neuron(
    contexts,
    family_params,
    x_grid,
    response_sd=response_sd_default,
):
    """
    likelihood_packages[used_context][true_context]
    = package for p(r | true_context, encoder_used_on_that_trial)
    """
    likelihood_packages = {}

    for used_k, used_params in family_params.items():
        likelihood_packages[used_k] = {}

        for true_k, context in contexts.items():
            likelihood_packages[used_k][true_k] = posterior_stats_one_neuron(
                context=context,
                a=used_params[0],
                b=used_params[1],
                x_grid=x_grid,
                response_sd=response_sd,
            )

    return likelihood_packages


def build_static_package_one_neuron(
    contexts,
    static_params,
    x_grid,
    response_sd=response_sd_default,
):
    px_mix = mixture_prior_pmf(contexts, x_grid)

    return posterior_stats_from_prior_one_neuron(
        x_grid=x_grid,
        px=px_mix,
        a=static_params[0],
        b=static_params[1],
        response_sd=response_sd,
    )

In [ ]:
# ---------- switching simulation + evaluation ----------

def simulate_context_sequence(n_steps, p_switch=0.01, seed=0):
    rng_local = np.random.default_rng(seed)
    context_seq = np.zeros(n_steps, dtype=int)
    context_seq[0] = rng_local.integers(0, 2)

    for t in range(1, n_steps):
        if rng_local.random() < p_switch:
            context_seq[t] = 1 - context_seq[t - 1]
        else:
            context_seq[t] = context_seq[t - 1]

    return context_seq


def get_mismatch_episode_lengths(df_eval):
    mismatch = df_eval["mismatch"].fillna(False).to_numpy(dtype=bool)

    lengths = []
    current = 0

    for val in mismatch:
        if val:
            current += 1
        elif current > 0:
            lengths.append(current)
            current = 0

    if current > 0:
        lengths.append(current)

    return lengths


def run_strategy_one_neuron(
    strategy_name,
    contexts,
    x_grid,
    local_params,
    static_params,
    target_params=None,
    gamma=0.0,
    p_switch=p_switch_main,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
    seed=0,
):
    rng_local = np.random.default_rng(seed)

    context_seq = simulate_context_sequence(
        n_steps=n_steps,
        p_switch=p_switch,
        seed=seed + 1000,
    )

    local_packages = build_local_packages_one_neuron(
        contexts=contexts,
        local_params=local_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    if target_params is None:
        target_params = static_params

    family_params, family_packages = build_family_packages_one_neuron(
        contexts=contexts,
        local_params=local_params,
        target_params=target_params,
        gamma=gamma,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    context_likelihood_packages = build_context_likelihood_packages_one_neuron(
        contexts=contexts,
        family_params=family_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    static_package = build_static_package_one_neuron(
        contexts=contexts,
        static_params=static_params,
        x_grid=x_grid,
        response_sd=response_sd,
    )

    transition_matrix = np.array([
        [1 - p_switch, p_switch],
        [p_switch, 1 - p_switch],
    ], dtype=float)

    belief = np.array([0.5, 0.5], dtype=float)

    rows = []

    for t in range(n_steps):
        true_context = int(context_seq[t])
        context_now = contexts[true_context]

        x_t = rng_local.normal(context_now["mu"], context_now["sigma"])

        if strategy_name == "static":
            used_params = static_params
            used_package = static_package
            estimated_context = -1
            belief_pred = belief.copy()

        elif strategy_name == "oracle":
            used_params = local_params[true_context]
            used_package = local_packages[true_context]
            estimated_context = true_context
            belief_pred = belief.copy()

        elif strategy_name == "adaptive":
            belief_pred = transition_matrix.T @ belief
            belief_pred /= belief_pred.sum()

            estimated_context = int(np.argmax(belief_pred))
            used_params = family_params[estimated_context]
            used_package = family_packages[estimated_context]

        else:
            raise ValueError("strategy_name must be 'static', 'oracle', or 'adaptive'")

        r_t = rng_local.normal(
            logistic(x_t, used_params[0], used_params[1]),
            response_sd,
        )

        xhat_t = interp_from_package(r_t, used_package, "posterior_mean")
        sq_error_t = (xhat_t - x_t) ** 2

        if strategy_name == "adaptive":
            likelihoods = np.array([
                predictive_density(
                    r_t,
                    context_likelihood_packages[estimated_context][0],
                    response_sd=response_sd,
                ),
                predictive_density(
                    r_t,
                    context_likelihood_packages[estimated_context][1],
                    response_sd=response_sd,
                ),
            ])

            belief = likelihoods * belief_pred
            belief_sum = belief.sum()

            if belief_sum <= 0:
                belief = np.array([0.5, 0.5], dtype=float)
            else:
                belief /= belief_sum

        elif strategy_name == "oracle":
            belief = np.eye(2)[true_context]

        rows.append({
            "t": t,
            "true_context": true_context,
            "estimated_context": estimated_context,
            "x": x_t,
            "r": r_t,
            "xhat": xhat_t,
            "sq_error": sq_error_t,
            "mismatch": (estimated_context != true_context) if estimated_context >= 0 else np.nan,
            "belief_context1": belief[1],
        })

    df = pd.DataFrame(rows)
    df_eval = df.iloc[burn_in:].copy()

    out = {
        "strategy": strategy_name,
        "gamma": gamma,
        "p_switch": p_switch,
        "global_mse": df_eval["sq_error"].mean(),
        "df": df,
    }

    if strategy_name == "adaptive":
        matched = df_eval.loc[~df_eval["mismatch"], "sq_error"]
        mismatched = df_eval.loc[df_eval["mismatch"], "sq_error"]
        mismatch_lengths = get_mismatch_episode_lengths(df_eval)

        out["mismatch_rate"] = df_eval["mismatch"].mean()
        out["matched_mse"] = matched.mean() if len(matched) > 0 else np.nan
        out["mismatched_mse"] = mismatched.mean() if len(mismatched) > 0 else np.nan
        out["mean_mismatch_duration"] = np.mean(mismatch_lengths) if len(mismatch_lengths) > 0 else 0.0

    return out


def sweep_gamma_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    target_params,
    family_name,
    gamma_grid,
    p_switch=p_switch_main,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    for gamma in gamma_grid:
        for seed_now in seeds:
            out = run_strategy_one_neuron(
                strategy_name="adaptive",
                contexts=contexts,
                x_grid=x_grid,
                local_params=local_params,
                static_params=static_params,
                target_params=target_params,
                gamma=gamma,
                p_switch=p_switch,
                n_steps=n_steps,
                burn_in=burn_in,
                response_sd=response_sd,
                seed=seed_now,
            )

            rows.append({
                "family": family_name,
                "gamma": gamma,
                "seed": seed_now,
                "global_mse": out["global_mse"],
                "mismatch_rate": out["mismatch_rate"],
                "matched_mse": out["matched_mse"],
                "mismatched_mse": out["mismatched_mse"],
                "mean_mismatch_duration": out["mean_mismatch_duration"],
            })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["family", "gamma"], as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
            mismatch_rate_mean=("mismatch_rate", "mean"),
            matched_mse_mean=("matched_mse", "mean"),
            mismatched_mse_mean=("mismatched_mse", "mean"),
            mean_mismatch_duration_mean=("mean_mismatch_duration", "mean"),
        )
        .sort_values(["family", "gamma"])
        .reset_index(drop=True)
    )

    return df, summary


def evaluate_strategy_set_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    infer_params,
    best_gamma_static,
    best_gamma_infer,
    infer_acc_params=None,
    best_gamma_infer_acc=None,
    nonlocal_params=None,
    best_gamma_nonlocal=None,
    p_switch=p_switch_main,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    strategy_specs = [
        ("static", None, None),
        ("oracle", None, None),
        ("adaptive_local", static_params, 0.0),
        ("adaptive_best_static", static_params, best_gamma_static),
        ("adaptive_best_infer_entropy", infer_params, best_gamma_infer),
    ]

    if infer_acc_params is not None and best_gamma_infer_acc is not None:
        strategy_specs.append(
            ("adaptive_best_infer_accuracy", infer_acc_params, best_gamma_infer_acc)
        )

    if nonlocal_params is not None and best_gamma_nonlocal is not None:
        strategy_specs.append(
            ("adaptive_best_nonlocal", nonlocal_params, best_gamma_nonlocal)
        )

    for seed_now in seeds:
        for label, target_params, gamma in strategy_specs:
            if label == "static":
                out = run_strategy_one_neuron(
                    strategy_name="static",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=static_params,
                    gamma=0.0,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            elif label == "oracle":
                out = run_strategy_one_neuron(
                    strategy_name="oracle",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=static_params,
                    gamma=0.0,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            else:
                out = run_strategy_one_neuron(
                    strategy_name="adaptive",
                    contexts=contexts,
                    x_grid=x_grid,
                    local_params=local_params,
                    static_params=static_params,
                    target_params=target_params,
                    gamma=gamma,
                    p_switch=p_switch,
                    n_steps=n_steps,
                    burn_in=burn_in,
                    response_sd=response_sd,
                    seed=seed_now,
                )

            rows.append({
                "strategy": label,
                "seed": seed_now,
                "global_mse": out["global_mse"],
                "mismatch_rate": out.get("mismatch_rate", np.nan),
                "matched_mse": out.get("matched_mse", np.nan),
                "mismatched_mse": out.get("mismatched_mse", np.nan),
                "mean_mismatch_duration": out.get("mean_mismatch_duration", np.nan),
            })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby("strategy", as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
            mismatch_rate_mean=("mismatch_rate", "mean"),
            matched_mse_mean=("matched_mse", "mean"),
            mismatched_mse_mean=("mismatched_mse", "mean"),
            mean_mismatch_duration_mean=("mean_mismatch_duration", "mean"),
        )
        .sort_values("global_mse_mean")
        .reset_index(drop=True)
    )

    return df, summary


def sweep_switch_prob_one_neuron(
    contexts,
    x_grid,
    local_params,
    static_params,
    infer_params,
    best_gamma_static,
    best_gamma_infer,
    p_switch_grid,
    infer_acc_params=None,
    best_gamma_infer_acc=None,
    nonlocal_params=None,
    best_gamma_nonlocal=None,
    seeds=seed_list,
    n_steps=n_steps_main,
    burn_in=burn_in_main,
    response_sd=response_sd_default,
):
    rows = []

    strategy_specs = [
        ("static", None, None),
        ("oracle", None, None),
        ("adaptive_local", static_params, 0.0),
        ("adaptive_best_static", static_params, best_gamma_static),
        ("adaptive_best_infer_entropy", infer_params, best_gamma_infer),
    ]

    if infer_acc_params is not None and best_gamma_infer_acc is not None:
        strategy_specs.append(
            ("adaptive_best_infer_accuracy", infer_acc_params, best_gamma_infer_acc)
        )

    if nonlocal_params is not None and best_gamma_nonlocal is not None:
        strategy_specs.append(
            ("adaptive_best_nonlocal", nonlocal_params, best_gamma_nonlocal)
        )

    for p_switch in p_switch_grid:
        for seed_now in seeds:
            for label, target_params, gamma in strategy_specs:
                if label == "static":
                    out = run_strategy_one_neuron(
                        strategy_name="static",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=static_params,
                        gamma=0.0,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                elif label == "oracle":
                    out = run_strategy_one_neuron(
                        strategy_name="oracle",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=static_params,
                        gamma=0.0,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                else:
                    out = run_strategy_one_neuron(
                        strategy_name="adaptive",
                        contexts=contexts,
                        x_grid=x_grid,
                        local_params=local_params,
                        static_params=static_params,
                        target_params=target_params,
                        gamma=gamma,
                        p_switch=p_switch,
                        n_steps=n_steps,
                        burn_in=burn_in,
                        response_sd=response_sd,
                        seed=seed_now,
                    )

                rows.append({
                    "p_switch": p_switch,
                    "strategy": label,
                    "seed": seed_now,
                    "global_mse": out["global_mse"],
                })

    df = pd.DataFrame(rows)

    summary = (
        df.groupby(["p_switch", "strategy"], as_index=False)
        .agg(
            global_mse_mean=("global_mse", "mean"),
            global_mse_std=("global_mse", "std"),
        )
    )

    return df, summary

In [ ]:
# ---------- gamma sweeps: all target families ----------

family_label_map = {
    "shrink_to_static": "toward static target",
    "shrink_to_infer": "toward entropy target",
    "shrink_to_infer_acc": "toward accuracy target",
    "shrink_to_nonlocal": "toward non-local target",
}


def run_gamma_family_set(
    contexts,
    x_grid,
    local_params,
    static_params,
    infer_params,
    infer_acc_params,
    nonlocal_params,
    gamma_grid,
    p_switch,
):
    target_map = {
        "shrink_to_static": static_params,
        "shrink_to_infer": infer_params,
        "shrink_to_infer_acc": infer_acc_params,
        "shrink_to_nonlocal": nonlocal_params,
    }

    df_map = {}
    summary_map = {}

    for family_name, target_params in target_map.items():
        df_now, summary_now = sweep_gamma_one_neuron(
            contexts=contexts,
            x_grid=x_grid,
            local_params=local_params,
            static_params=static_params,
            target_params=target_params,
            family_name=family_name,
            gamma_grid=gamma_grid,
            p_switch=p_switch,
        )
        df_map[family_name] = df_now
        summary_map[family_name] = summary_now

    summary_all = pd.concat(summary_map.values(), ignore_index=True)

    best_gamma_map = {
        family_name: float(
            summary_now.loc[summary_now["global_mse_mean"].idxmin(), "gamma"]
        )
        for family_name, summary_now in summary_map.items()
    }

    best_row = summary_all.loc[summary_all["global_mse_mean"].idxmin()]
    best_family = best_row["family"]
    best_gamma_overall = float(best_row["gamma"])
    best_target_params = target_map[best_family]

    return df_map, summary_map, summary_all, best_gamma_map, best_family, best_gamma_overall, best_target_params


# mean-switching
(
    gamma_df_map_mean,
    gamma_summary_map_mean,
    gamma_summary_mean,
    best_gamma_map_mean,
    best_family_mean,
    best_gamma_mean_overall,
    best_target_params_mean,
) = run_gamma_family_set(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    infer_acc_params=infer_params_mean_acc,
    nonlocal_params=nonlocal_target_params_mean,
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_summary_mean_static = gamma_summary_map_mean["shrink_to_static"]
gamma_summary_mean_infer = gamma_summary_map_mean["shrink_to_infer"]
gamma_summary_mean_infer_acc = gamma_summary_map_mean["shrink_to_infer_acc"]
gamma_summary_mean_nonlocal = gamma_summary_map_mean["shrink_to_nonlocal"]

best_gamma_mean_static = best_gamma_map_mean["shrink_to_static"]
best_gamma_mean_infer = best_gamma_map_mean["shrink_to_infer"]
best_gamma_mean_infer_acc = best_gamma_map_mean["shrink_to_infer_acc"]
best_gamma_mean_nonlocal = best_gamma_map_mean["shrink_to_nonlocal"]

best_target_label_mean = family_label_map[best_family_mean]

# variance-switching
(
    gamma_df_map_var,
    gamma_summary_map_var,
    gamma_summary_var,
    best_gamma_map_var,
    best_family_var,
    best_gamma_var_overall,
    best_target_params_var,
) = run_gamma_family_set(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    infer_acc_params=infer_params_var_acc,
    nonlocal_params=nonlocal_target_params_var,
    gamma_grid=gamma_grid,
    p_switch=p_switch_main,
)

gamma_summary_var_static = gamma_summary_map_var["shrink_to_static"]
gamma_summary_var_infer = gamma_summary_map_var["shrink_to_infer"]
gamma_summary_var_infer_acc = gamma_summary_map_var["shrink_to_infer_acc"]
gamma_summary_var_nonlocal = gamma_summary_map_var["shrink_to_nonlocal"]

best_gamma_var_static = best_gamma_map_var["shrink_to_static"]
best_gamma_var_infer = best_gamma_map_var["shrink_to_infer"]
best_gamma_var_infer_acc = best_gamma_map_var["shrink_to_infer_acc"]
best_gamma_var_nonlocal = best_gamma_map_var["shrink_to_nonlocal"]

best_target_label_var = family_label_map[best_family_var]

print("best gamma (mean, static):", best_gamma_mean_static)
print("best gamma (mean, entropy target):", best_gamma_mean_infer)
print("best gamma (mean, accuracy target):", best_gamma_mean_infer_acc)
print("best gamma (mean, non-local target):", best_gamma_mean_nonlocal)
print("overall best mean family:", best_target_label_mean, "| gamma =", best_gamma_mean_overall)
display(gamma_summary_mean)

print("best gamma (variance, static):", best_gamma_var_static)
print("best gamma (variance, entropy target):", best_gamma_var_infer)
print("best gamma (variance, accuracy target):", best_gamma_var_infer_acc)
print("best gamma (variance, non-local target):", best_gamma_var_nonlocal)
print("overall best variance family:", best_target_label_var, "| gamma =", best_gamma_var_overall)
display(gamma_summary_var)

In [ ]:
# ---------- entropy-target vs accuracy-target gamma curves ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].errorbar(
    gamma_summary_mean_infer["gamma"],
    gamma_summary_mean_infer["global_mse_mean"],
    yerr=gamma_summary_mean_infer["global_mse_std"],
    marker="o",
    capsize=3,
    label="entropy target",
)
axes[0].errorbar(
    gamma_summary_mean_infer_acc["gamma"],
    gamma_summary_mean_infer_acc["global_mse_mean"],
    yerr=gamma_summary_mean_infer_acc["global_mse_std"],
    marker="o",
    capsize=3,
    label="accuracy target",
)
axes[0].set_title("mean-switching: infer-family gamma sweep")
axes[0].set_xlabel("gamma")
axes[0].set_ylabel("global MSE")
axes[0].legend()

axes[1].errorbar(
    gamma_summary_var_infer["gamma"],
    gamma_summary_var_infer["global_mse_mean"],
    yerr=gamma_summary_var_infer["global_mse_std"],
    marker="o",
    capsize=3,
    label="entropy target",
)
axes[1].errorbar(
    gamma_summary_var_infer_acc["gamma"],
    gamma_summary_var_infer_acc["global_mse_mean"],
    yerr=gamma_summary_var_infer_acc["global_mse_std"],
    marker="o",
    capsize=3,
    label="accuracy target",
)
axes[1].set_title("variance-switching: infer-family gamma sweep")
axes[1].set_xlabel("gamma")
axes[1].set_ylabel("global MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- main strategy comparison ----------

main_df_mean, main_summary_mean = evaluate_strategy_set_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    best_gamma_static=best_gamma_mean_static,
    best_gamma_infer=best_gamma_mean_infer,
    infer_acc_params=infer_params_mean_acc,
    best_gamma_infer_acc=best_gamma_mean_infer_acc,
    nonlocal_params=nonlocal_target_params_mean,
    best_gamma_nonlocal=best_gamma_mean_nonlocal,
    p_switch=p_switch_main,
)

main_df_var, main_summary_var = evaluate_strategy_set_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    best_gamma_static=best_gamma_var_static,
    best_gamma_infer=best_gamma_var_infer,
    infer_acc_params=infer_params_var_acc,
    best_gamma_infer_acc=best_gamma_var_infer_acc,
    nonlocal_params=nonlocal_target_params_var,
    best_gamma_nonlocal=best_gamma_var_nonlocal,
    p_switch=p_switch_main,
)

print("main strategy comparison: mean-switching")
display(main_summary_mean)

print("main strategy comparison: variance-switching")
display(main_summary_var)

In [ ]:
# ---------- fitted nonlinearities plot ----------

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# mean-switching
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[0][0], local_params_mean[0][1]),
    label="local context 0",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, local_params_mean[1][0], local_params_mean[1][1]),
    label="local context 1",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, static_params_mean[0], static_params_mean[1]),
    label="static target",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean[0], infer_params_mean[1]),
    label="entropy target",
)
axes[0].plot(
    x_grid_mean,
    logistic(x_grid_mean, infer_params_mean_acc[0], infer_params_mean_acc[1]),
    linestyle="--",
    label="accuracy target",
)
axes[0].set_title("mean-switching: fitted nonlinearities")
axes[0].set_xlabel("stimulus x")
axes[0].set_ylabel("response")
axes[0].legend()

# variance-switching
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[0][0], local_params_var[0][1]),
    label="local context 0",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, local_params_var[1][0], local_params_var[1][1]),
    label="local context 1",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, static_params_var[0], static_params_var[1]),
    label="static target",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var[0], infer_params_var[1]),
    label="entropy target",
)
axes[1].plot(
    x_grid_var,
    logistic(x_grid_var, infer_params_var_acc[0], infer_params_var_acc[1]),
    linestyle="--",
    label="accuracy target",
)
axes[1].set_title("variance-switching: fitted nonlinearities")
axes[1].set_xlabel("stimulus x")
axes[1].set_ylabel("response")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- gamma sweep plots ----------

plot_specs = [
    ("toward static target",    gamma_summary_mean_static,    gamma_summary_var_static),
    ("toward entropy target",   gamma_summary_mean_infer,     gamma_summary_var_infer),
    ("toward accuracy target",  gamma_summary_mean_infer_acc, gamma_summary_var_infer_acc),
    ("toward non-local target", gamma_summary_mean_nonlocal,  gamma_summary_var_nonlocal),
]

mean_static_baseline = gamma_summary_mean_static.loc[
    np.isclose(gamma_summary_mean_static["gamma"], 1.0),
    "global_mse_mean"
].iloc[0]

var_static_baseline = gamma_summary_var_static.loc[
    np.isclose(gamma_summary_var_static["gamma"], 1.0),
    "global_mse_mean"
].iloc[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# mean-switching: global MSE
for label, mean_summary, _ in plot_specs:
    axes[0, 0].errorbar(
        mean_summary["gamma"],
        mean_summary["global_mse_mean"],
        yerr=mean_summary["global_mse_std"],
        marker="o",
        capsize=3,
        label=label,
    )

    best_idx = mean_summary["global_mse_mean"].idxmin()
    axes[0, 0].scatter(
        mean_summary.loc[best_idx, "gamma"],
        mean_summary.loc[best_idx, "global_mse_mean"],
        s=60,
        zorder=5,
    )

axes[0, 0].axhline(
    mean_static_baseline,
    linestyle="--",
    linewidth=1.5,
    label="static-code level",
)
axes[0, 0].set_title("mean-switching: global MSE vs gamma")
axes[0, 0].set_xlabel("gamma (bias toward target)")
axes[0, 0].set_ylabel("global MSE")
axes[0, 0].legend()

# variance-switching: global MSE
for label, _, var_summary in plot_specs:
    axes[0, 1].errorbar(
        var_summary["gamma"],
        var_summary["global_mse_mean"],
        yerr=var_summary["global_mse_std"],
        marker="o",
        capsize=3,
        label=label,
    )

    best_idx = var_summary["global_mse_mean"].idxmin()
    axes[0, 1].scatter(
        var_summary.loc[best_idx, "gamma"],
        var_summary.loc[best_idx, "global_mse_mean"],
        s=60,
        zorder=5,
    )

axes[0, 1].axhline(
    var_static_baseline,
    linestyle="--",
    linewidth=1.5,
    label="static-code level",
)
axes[0, 1].set_title("variance-switching: global MSE vs gamma")
axes[0, 1].set_xlabel("gamma (bias toward target)")
axes[0, 1].set_ylabel("global MSE")
axes[0, 1].legend()

# mean-switching: mismatch rate
for label, mean_summary, _ in plot_specs:
    axes[1, 0].plot(
        mean_summary["gamma"],
        mean_summary["mismatch_rate_mean"],
        marker="o",
        label=label,
    )

axes[1, 0].set_title("mean-switching: mismatch rate vs gamma")
axes[1, 0].set_xlabel("gamma")
axes[1, 0].set_ylabel("mismatch rate")
axes[1, 0].legend()

# variance-switching: mismatch rate
for label, _, var_summary in plot_specs:
    axes[1, 1].plot(
        var_summary["gamma"],
        var_summary["mismatch_rate_mean"],
        marker="o",
        label=label,
    )

axes[1, 1].set_title("variance-switching: mismatch rate vs gamma")
axes[1, 1].set_xlabel("gamma")
axes[1, 1].set_ylabel("mismatch rate")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- volatility sweep ----------

p_switch_grid = [0.002, 0.005, 0.01, 0.02, 0.04, 0.08]

switch_df_mean, switch_summary_mean = sweep_switch_prob_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    best_gamma_static=best_gamma_mean_static,
    best_gamma_infer=best_gamma_mean_infer,
    infer_acc_params=infer_params_mean_acc,
    best_gamma_infer_acc=best_gamma_mean_infer_acc,
    nonlocal_params=nonlocal_target_params_mean,
    best_gamma_nonlocal=best_gamma_mean_nonlocal,
    p_switch_grid=p_switch_grid,
)

switch_df_var, switch_summary_var = sweep_switch_prob_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    best_gamma_static=best_gamma_var_static,
    best_gamma_infer=best_gamma_var_infer,
    infer_acc_params=infer_params_var_acc,
    best_gamma_infer_acc=best_gamma_var_infer_acc,
    nonlocal_params=nonlocal_target_params_var,
    best_gamma_nonlocal=best_gamma_var_nonlocal,
    p_switch_grid=p_switch_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for strategy in switch_summary_mean["strategy"].unique():
    sub = switch_summary_mean[switch_summary_mean["strategy"] == strategy]
    axes[0].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[0].set_title("mean-switching: global MSE vs switch probability")
axes[0].set_xlabel("switch probability")
axes[0].set_ylabel("global MSE")
axes[0].legend()

for strategy in switch_summary_var["strategy"].unique():
    sub = switch_summary_var[switch_summary_var["strategy"] == strategy]
    axes[1].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[1].set_title("variance-switching: global MSE vs switch probability")
axes[1].set_xlabel("switch probability")
axes[1].set_ylabel("global MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- volatility sweep ----------

p_switch_grid = [0.002, 0.005, 0.01, 0.02, 0.04, 0.08]

switch_df_mean, switch_summary_mean = sweep_switch_prob_one_neuron(
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    infer_params=infer_params_mean,
    best_gamma_static=best_gamma_mean_static,
    best_gamma_infer=best_gamma_mean_infer,
    infer_acc_params=infer_params_mean_acc,
    best_gamma_infer_acc=best_gamma_mean_infer_acc,
    nonlocal_params=nonlocal_target_params_mean,
    best_gamma_nonlocal=best_gamma_mean_nonlocal,
    p_switch_grid=p_switch_grid,
)

switch_df_var, switch_summary_var = sweep_switch_prob_one_neuron(
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    infer_params=infer_params_var,
    best_gamma_static=best_gamma_var_static,
    best_gamma_infer=best_gamma_var_infer,
    infer_acc_params=infer_params_var_acc,
    best_gamma_infer_acc=best_gamma_var_infer_acc,
    nonlocal_params=nonlocal_target_params_var,
    best_gamma_nonlocal=best_gamma_var_nonlocal,
    p_switch_grid=p_switch_grid,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for strategy in switch_summary_mean["strategy"].unique():
    sub = switch_summary_mean[switch_summary_mean["strategy"] == strategy]
    axes[0].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[0].set_title("mean-switching: global MSE vs switch probability")
axes[0].set_xlabel("switch probability")
axes[0].set_ylabel("global MSE")
axes[0].legend()

for strategy in switch_summary_var["strategy"].unique():
    sub = switch_summary_var[switch_summary_var["strategy"] == strategy]
    axes[1].plot(sub["p_switch"], sub["global_mse_mean"], marker="o", label=strategy)

axes[1].set_title("variance-switching: global MSE vs switch probability")
axes[1].set_xlabel("switch probability")
axes[1].set_ylabel("global MSE")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---------- representative adaptive traces ----------

trace_mean_local = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=static_params_mean,
    gamma=0.0,
    p_switch=p_switch_main,
    seed=123,
)

trace_mean_best = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_mean,
    x_grid=x_grid_mean,
    local_params=local_params_mean,
    static_params=static_params_mean,
    target_params=best_target_params_mean,
    gamma=best_gamma_mean_overall,
    p_switch=p_switch_main,
    seed=123,
)

trace_var_local = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=static_params_var,
    gamma=0.0,
    p_switch=p_switch_main,
    seed=123,
)

trace_var_best = run_strategy_one_neuron(
    strategy_name="adaptive",
    contexts=contexts_var,
    x_grid=x_grid_var,
    local_params=local_params_var,
    static_params=static_params_var,
    target_params=best_target_params_var,
    gamma=best_gamma_var_overall,
    p_switch=p_switch_main,
    seed=123,
)

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)

show_slice = slice(500, 900)

axes[0, 0].plot(
    trace_mean_local["df"].iloc[show_slice]["t"],
    trace_mean_local["df"].iloc[show_slice]["true_context"],
    label="true context",
)
axes[0, 0].plot(
    trace_mean_local["df"].iloc[show_slice]["t"],
    trace_mean_local["df"].iloc[show_slice]["estimated_context"],
    label="estimated context",
)
axes[0, 0].set_title("mean-switching: naive local adaptive")
axes[0, 0].set_ylabel("context")
axes[0, 0].legend()

axes[0, 1].plot(
    trace_mean_best["df"].iloc[show_slice]["t"],
    trace_mean_best["df"].iloc[show_slice]["true_context"],
    label="true context",
)
axes[0, 1].plot(
    trace_mean_best["df"].iloc[show_slice]["t"],
    trace_mean_best["df"].iloc[show_slice]["estimated_context"],
    label="estimated context",
)
axes[0, 1].set_title(f"mean-switching: best adaptive\n{best_target_label_mean}, gamma={best_gamma_mean_overall:.2f}")
axes[0, 1].set_ylabel("context")
axes[0, 1].legend()

axes[1, 0].plot(
    trace_var_local["df"].iloc[show_slice]["t"],
    trace_var_local["df"].iloc[show_slice]["true_context"],
    label="true context",
)
axes[1, 0].plot(
    trace_var_local["df"].iloc[show_slice]["t"],
    trace_var_local["df"].iloc[show_slice]["estimated_context"],
    label="estimated context",
)
axes[1, 0].set_title("variance-switching: naive local adaptive")
axes[1, 0].set_xlabel("time step")
axes[1, 0].set_ylabel("context")
axes[1, 0].legend()

axes[1, 1].plot(
    trace_var_best["df"].iloc[show_slice]["t"],
    trace_var_best["df"].iloc[show_slice]["true_context"],
    label="true context",
)
axes[1, 1].plot(
    trace_var_best["df"].iloc[show_slice]["t"],
    trace_var_best["df"].iloc[show_slice]["estimated_context"],
    label="estimated context",
)
axes[1, 1].set_title(f"variance-switching: best adaptive\n{best_target_label_var}, gamma={best_gamma_var_overall:.2f}")
axes[1, 1].set_xlabel("time step")
axes[1, 1].set_ylabel("context")
axes[1, 1].legend()

plt.tight_layout()
plt.show()